<a href="https://colab.research.google.com/github/KayoLage/Ferramenta-SoftPipeline-INF450/blob/main/SoftPipe_Tool_INF450_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Trabalho INF450 - Ferramenta didática para Soft. Pipeline**

### Exemplo usado para debug
```
loop: ld f1,0(r1)
mult f2,f1,f1
ld f3,4(r1)
mult f3,f3,f2
mult f3,f3,f1
add f3,f3,f2
sd f3,0(r1)
addi r1,r1,8
bne r1,r2,loop
```

### Mais exemplos para debug:

```
LOOP: Ld f3,0(r1)
add f3,f3,f3
mult f3,f3,f3
sd f3,0(r1)
Addi r1,r1,8
bne r1,r2,LOOP
```

---

```
LOOP:Ld f3,0(r1)
multf f1,f2,f3
Ld f4,4(r1)
mulf f5,f1,f4
mult f4,f4,f2
Sd f5,0(r1)
addf f5,f5,f4
Sd f5,4(r1)
Addi r1,r1,8
Bne r1,r2,LOOP
```

```
loop: ld f1,0(r1)
ld f2,4(r1)
add f1,f1,f2
add f4,f2,f3
mul f1,f1,f2
mul f4,f2,f4
sd f1,0(r1)
sd f4,4(r1)
addi r1,r1,8
bne r1,r5, loop
```

---
## Utilitários, imports, funções e classes auxiliares

### Imports

In [ ]:
import re
import copy
import graphviz
import networkx as nx
import ipywidgets as widgets
from tabulate import tabulate
import matplotlib.pyplot as plt
from collections import defaultdict
from matplotlib.patches import ConnectionStyle
from IPython.display import display, clear_output
from matplotlib.patches import Ellipse, FancyArrowPatch
from matplotlib.offsetbox import TextArea, HPacker, AnnotationBbox

from matplotlib._pylab_helpers import Gcf
Gcf.figs.clear()

### Classes e funções auxiliares

In [ ]:
def _ensure_mem_size(mem_list, idx):
    """Auxiliar para expandir a memória dinamicamente caso o índice vá além do limite atual"""
    if idx >= len(mem_list):
        mem_list.extend([float(i) for i in range(len(mem_list), idx + 1)])

class MachineConfig:
    def __init__(self, mem_size=200, word_size=4, forced_loops=30):
        self.mem_size = mem_size
        self.word_size = word_size
        self.forced_loops = forced_loops

        # Gera automaticamente r1=1, r2=2... e f1=1.0, f2=2.0... até 31
        self.reg_init = {}
        for i in range(1, 32):
            self.reg_init[f"r{i}"] = i
            self.reg_init[f"f{i}"] = float(i)

    def fresh_memory(self):
        return [float(x) for x in range(self.mem_size)]

    def fresh_registers(self):
        return dict(self.reg_init)

    def __repr__(self):
        return f"MachineConfig(mem_size={self.mem_size}, word_size={self.word_size}, forced_loops={self.forced_loops})"


class ISASimulator:
    def __init__(self, config: MachineConfig):
        self.config = config

    @staticmethod
    def _get_reg(name, R, F):
        return F.setdefault(name, 0.0) if name.startswith('f') else R.setdefault(name, 0)

    @staticmethod
    def _set_reg(name, val, R, F):
        if name.startswith('f'):
            F[name] = val
        else:
            R[name] = val

    def execute_straightline(self, instr, R, F, mem):
        """Executa UMA instrução não-branch/jump/label com auto-expansão de memória."""
        cat, ops, op = instr['category'], instr['operands'], instr['op']
        ws = self.config.word_size

        if cat == 'r_type':
            dest, s1, s2 = ops
            a, b = self._get_reg(s1, R, F), self._get_reg(s2, R, F)
            val = a * b if op == '*' else (a + b if op == '+' else a - b)
            self._set_reg(dest, val, R, F)
        elif cat == 'i_type':
            dest, src, imm = ops
            a = self._get_reg(src, R, F)
            val = a + int(imm) if op == '+' else a - int(imm)
            self._set_reg(dest, val, R, F)
        elif cat == 'load':
            dest, mem_op = ops
            offset, base = MEM_RE.match(mem_op).groups()
            addr = self._get_reg(base, R, F) + int(offset)
            idx = addr // ws
            _ensure_mem_size(mem, idx)
            self._set_reg(dest, mem[idx], R, F)
        elif cat == 'store':
            src, mem_op = ops
            offset, base = MEM_RE.match(mem_op).groups()
            addr = self._get_reg(base, R, F) + int(offset)
            idx = addr // ws
            _ensure_mem_size(mem, idx)
            mem[idx] = self._get_reg(src, R, F)
        elif cat == 'mov':
            dest, src = ops
            self._set_reg(dest, self._get_reg(src, R, F), R, F)
        else:
            raise ValueError(f"categoria '{cat}' não é executável diretamente.")

    @staticmethod
    def build_label_map(instructions):
        labels = {}
        for i, instr in enumerate(instructions):
            if instr.get('type') == 'label' or instr.get('label'):
                labels[instr['label']] = i
        return labels

    def run_full_program(self, instructions, R_init=None, F_init=None, mem=None, max_steps=500_000):
        R = dict(R_init) if R_init is not None else self.config.fresh_registers()
        # FIX DEFINITIVO: Inicializa F com os valores flutuantes reais da máquina (f1=1.0, f2=2.0...)
        F = dict(F_init) if F_init is not None else {k: v for k, v in self.config.fresh_registers().items() if k.startswith('f')}
        mem = mem if mem is not None else self.config.fresh_memory()
        labels = self.build_label_map(instructions)
        pc, steps = 0, 0
        label_visits = defaultdict(int)
        n = len(instructions)

        while 0 <= pc < n and steps < max_steps:
            instr = instructions[pc]
            steps += 1

            if instr.get('type') == 'label':
                pc += 1
                continue

            if instr.get('label'):
                label_visits[instr['label']] += 1

            cat = instr['category']
            if cat == 'branch':
                s1, s2, target = instr['operands']
                a, b, op = self._get_reg(s1, R, F), self._get_reg(s2, R, F), instr['op']
                taken = {'!=': a != b, '==': a == b, '<': a < b,
                         '>': a > b, '<=': a <= b, '>=': a >= b}[op]

                if taken and target in label_visits and label_visits[target] >= self.config.forced_loops:
                    taken = False

                pc = labels[target] if taken else pc + 1

            elif cat == 'jump':
                (target,) = instr['operands']
                if target in label_visits and label_visits[target] >= self.config.forced_loops:
                    pc = pc + 1
                else:
                    pc = labels[target]
            else:
                self.execute_straightline(instr, R, F, mem)
                pc += 1

        return R, F, mem, label_visits

    @staticmethod
    def detect_induction_strides(instructions):
        strides = {}
        for instr in instructions:
            if instr['category'] == 'i_type':
                dest, src, imm = instr['operands']
                if dest == src and not dest.startswith('f'):
                    strides[dest] = int(imm) if instr['op'] == '+' else -int(imm)
        return strides

    @staticmethod
    def collect_written_regs(instructions):
        written = set()
        for instr in instructions:
            if instr.get('category') in ('r_type', 'i_type', 'load', 'mov'):
                written.add(instr['operands'][0])
        return written


class PipelineValidator:
    def __init__(self, original_instructions, config: MachineConfig):
        self.original_instructions = original_instructions
        self.config = config
        self.simulator = ISASimulator(config)

    def compute_reference(self):
        _, _, mem_final, label_visits = self.simulator.run_full_program(self.original_instructions)
        strides = self.simulator.detect_induction_strides(self.original_instructions)
        total_iterations = max(label_visits.values()) if label_visits else 1
        return mem_final, strides, total_iterations

    def simulate_pipeline(self, stages_user, strides, total_iterations):
        ws = self.config.word_size
        mem_pipe = self.config.fresh_memory()
        # FIX DEFINITIVO: Inicializa F_pipe com os valores flutuantes corretos do estado inicial
        F_pipe = {k: v for k, v in self.config.reg_init.items() if k.startswith('f')}
        max_lvl = max(stages_user.keys()) if stages_user else 0
        total_cycles = total_iterations + max_lvl

        for cycle in range(total_cycles):
            F_snap = dict(F_pipe)
            mem_snap = list(mem_pipe)
            pending_F, pending_mem = {}, {}

            for lvl in sorted(stages_user.keys(), reverse=True):
                it = cycle - lvl
                if not (0 <= it < total_iterations):
                    continue
                R_it = {reg: self.config.reg_init.get(reg, 0) + it * step for reg, step in strides.items()}
                for instr in stages_user[lvl]:
                    cat, ops, op = instr['category'], instr['operands'], instr['op']
                    if cat == 'load':
                        dest, mem_op = ops
                        offset, base = MEM_RE.match(mem_op).groups()
                        addr = R_it.get(base, 0) + int(offset)
                        idx = addr // ws

                        if idx >= len(mem_pipe):
                            _ensure_mem_size(mem_pipe, idx)
                        val = mem_snap[idx] if idx < len(mem_snap) else float(idx)
                        pending_F[dest] = val
                    elif cat == 'store':
                        src, mem_op = ops
                        offset, base = MEM_RE.match(mem_op).groups()
                        addr = R_it.get(base, 0) + int(offset)
                        idx = addr // ws
                        if idx >= len(mem_pipe):
                            _ensure_mem_size(mem_pipe, idx)
                        pending_mem[idx] = F_snap.get(src, pending_F.get(src, 0.0))
                    elif cat == 'mov':
                        dest, src = ops
                        pending_F[dest] = F_snap.get(src, pending_F.get(src, 0.0))
                    elif cat == 'r_type':
                        dest, s1, s2 = ops
                        a = F_snap.get(s1, pending_F.get(s1, 0.0))
                        b = F_snap.get(s2, pending_F.get(s2, 0.0))
                        pending_F[dest] = a * b if op == '*' else (a + b if op == '+' else a - b)
                    elif cat == 'i_type':
                        dest, src, imm = ops
                        a = F_snap.get(src, pending_F.get(src, 0.0)) if src.startswith('f') else R_it.get(src, 0)
                        pending_F[dest] = a + int(imm) if op == '+' else a - int(imm)

            F_pipe.update(pending_F)
            for idx, val in pending_mem.items():
                mem_pipe[idx] = val

        return mem_pipe

    def validate(self, user_canvas_nodes):
        if not user_canvas_nodes:
            return {"error": "O canvas está vazio! Monte um grafo antes de validar."}
        if not self.original_instructions:
            return {"error": "Nenhum programa original carregado para servir de referência."}

        try:
            mem_gabarito, strides, total_iterations = self.compute_reference()
        except Exception as e:
            return {"error": f"Erro executando o programa original: {e}"}

        stages_user = defaultdict(list)
        for item in user_canvas_nodes:
            stages_user[item['level']].append(parse_line(item['text'], 0))

        try:
            mem_pipe = self.simulate_pipeline(stages_user, strides, total_iterations)
        except Exception as e:
            return {"error": f"Erro na simulação do pipeline: {e}"}

        return {
            "success": mem_gabarito == mem_pipe,
            "mem_gabarito": mem_gabarito,
            "mem_pipe": mem_pipe,
            "strides": strides,
            "total_iterations": total_iterations,
            "error": None,
        }

### Classe para Tomasulo - Escalonamento Dinâmico

In [ ]:
class TomasuloEngineInteractive:
    def __init__(self, raw_lines):
        self.clk = 1
        self.stall = (None, None)
        self.done_branch = False
        self.stop = False
        self.forced_stop = False
        self.bne_predicted_inst = None
        self.prev_fwd = {}
        self.pipeline = {'fetch': None, 'decode': None}
        self.issue_ticket = 0
        self.stations_to_write = {}
        self.stations_to_reset = []

        # Latências de Hardware
        self.cicles = {'l': 2, 'a': 3, 'm': 5, 'i': 1, 'v': 1}  # 'v' = mov
        self.mov_counter = 0

        # Estações de Reserva
        self.stations = {}
        for op, cnt in [('l', 3), ('m', 2), ('a', 2), ('i', 1)]:
            for i in range(cnt):
                self.stations[f"{op}{i+1}"] = {
                    'inst': None, 'regs': [None]*3, 'end': None,
                    'fwd': None, 'write': None, 'branch': False, 'ticket': None
                }
        self.stage = {s: {'execute': None} for s in self.stations}

        # Banco de Registradores e Memória
        self.registers = {f'r{i}': i * 4 for i in range(1, 5)}
        self.registers.update({f'f{i}': float(i) for i in range(1, 13)})
        self.memory = [float(x) for x in range(32)]

        # Processamento e normalização da fila de instruções
        self.instructions = []
        seen_counts = defaultdict(int)
        for line in raw_lines:
            clean = line.split('#')[0].strip().lower()
            if not clean: continue
            if ":" in clean: clean = clean.split(":", 1)[1].strip()
            seen_counts[clean] += 1
            unique_name = f"{clean}   #{seen_counts[clean]}" if seen_counts[clean] > 1 else clean
            self.instructions.append(unique_name)

        self.branch_instructions = self.instructions.copy()
        self.x_headers = ["fetch", "decode", "execute", "write", "fetch_1", "decode_1", "execute_1", "write_1"]
        self.y_headers = self.instructions.copy()
        self.table_data = {y: {h: "" for header in self.x_headers for h in [header]} for y in self.y_headers}

    def parse_op(self, reg):
        try: return int(reg)
        except ValueError:
            val = self.registers.get(reg, 0.0)
            if isinstance(val, tuple): return val[0]
            return val

    def write_same_clk(self, this_id, delay):
        write = self.stations[this_id]['write']
        for id in [s for s in self.stations if s != this_id]:
            if this_id[0] == 'i' or id[0] == 'i': continue
            if (write and self.stations[id]['write']) and self.stations[id]['end'] == delay: return True
        return False

    def check_dependencies(self, id, op, regs):
        def get_val(r):
            v = self.registers.get(r, 0.0)
            return v[0] if isinstance(v, tuple) else v

        reg_0_val = get_val(regs[0])
        reg_1_val = None if op in ['sd','ld'] else get_val(regs[1])
        reg_2_val = self.parse_op(regs[2])
        delay = self.clk

        if reg_0_val is not None and reg_1_val is not None and reg_2_val is not None:
            if isinstance(reg_1_val, str) and self.stations[reg_1_val]['end']:
                delay = max(delay, self.stations[reg_1_val]['end'])
            if isinstance(reg_2_val, str) and self.stations[reg_2_val]['end']:
                delay = max(delay, self.stations[reg_2_val]['end'])

        self.stations[id]['write'] = False if op == 'sd' else True
        if op == 'sd' and isinstance(reg_0_val, str) and self.stations[reg_0_val]['end']:
            delay = max(delay, self.stations[reg_0_val]['end'])

        # ⚙️ CORREÇÃO: Define a latência baseada na unidade real (addi/bne = 1 ciclo, add/addf = 3 ciclos)
        if op in ['addi', 'subi', 'bne']:
            delay += self.cicles['i']
        elif op in ['add', 'addf', 'sub', 'subf']:
            delay += self.cicles['a']
        else:
            delay += self.cicles['l' if op == 'sd' else (op[0] if op[0] in self.cicles else 'i')]

        while self.write_same_clk(id, delay): delay += 1

        self.stations[id]['end'] = delay
        self.stations[id]['regs'][0] = reg_0_val if op == 'sd' else regs[0]
        self.stations[id]['regs'][1] = int(regs[1]) if op in ['sd','ld'] else reg_1_val
        self.stations[id]['regs'][2] = reg_2_val

        if op != 'sd':
            old_val = self.registers.get(regs[0], 0.0)
            if isinstance(old_val, tuple): old_val = old_val[1]
            self.registers[regs[0]] = (id, old_val, self.issue_ticket)
            self.stations[id]['ticket'] = self.issue_ticket
            self.issue_ticket += 1

    def check_dependencies_mov(self, id, dest, src):
        def get_val(r):
            v = self.registers.get(r, 0.0)
            return v[0] if isinstance(v, tuple) else v

        src_val = get_val(src)
        delay = self.clk

        if isinstance(src_val, str) and src_val in self.stations and self.stations[src_val]['end']:
            delay = max(delay, self.stations[src_val]['end'])

        self.stations[id]['write'] = True
        delay += self.cicles['v']

        while self.write_same_clk(id, delay): delay += 1

        self.stations[id]['end'] = delay
        self.stations[id]['regs'][0] = dest
        self.stations[id]['regs'][1] = src_val
        self.stations[id]['regs'][2] = None

        old_val = self.registers.get(dest, 0.0)
        if isinstance(old_val, tuple): old_val = old_val[1]
        self.registers[dest] = (id, old_val, self.issue_ticket)
        self.stations[id]['ticket'] = self.issue_ticket
        self.issue_ticket += 1

    def get_station(self, op, clk):
        op = op.lower()
        target_op = "i" if op in ["addi", "subi", "bne"] else ('l' if op == 'sd' else ('a' if op in ['add', 'addf'] else op[0]))
        for id in [s for s in self.stations if s.startswith(target_op)]:
            if self.stations[id]['end'] is None or self.stations[id]['end'] < clk: return id
        return None

    def next_available_station(self, op):
        free_clk = None
        # ⚙️ CORREÇÃO: Alinhamento com a separação estrutural de inteiros vs float adders
        target_op = "i" if op in ["addi", "subi"] else ('l' if op == 'sd' else ('a' if op in ['add', 'addf'] else op[0]))
        for id in [s for s in self.stations if s.startswith(target_op)]:
            if self.stations[id]['end']:
                free_clk = self.stations[id]['end'] if free_clk is None else min(free_clk, self.stations[id]['end'])
        return free_clk

    def reset_station(self, id):
        self.stations[id] = {'inst': None, 'regs': [None]*3, 'end': None, 'fwd': None, 'write': None, 'branch': False, 'ticket': None}

    def is_done(self):
        if self.forced_stop: return True
        return len(self.instructions) == 0 and self.pipeline['decode'] is None and all(self.stations[s]['end'] is None or self.stations[s]['end'] < self.clk for s in self.stations) and len(self.stations_to_write) == 0

    def step_cycle(self):
        if self.is_done(): return

        # WRITE
        for id in list(self.stations_to_write.keys()):
            if self.stations_to_write[id]['end'] is None or self.stations_to_write[id]['end'] + 1 != self.clk: continue
            tagw = 'write_1' if self.stations_to_write[id]['branch'] else 'write'
            inst = self.stations_to_write[id]['inst']

            target_key = inst
            for k in self.table_data:
                if k.startswith(inst): target_key = k; break

            current_content = self.table_data[target_key][tagw]
            if not inst.startswith('sd') and not inst.startswith('bne'):
                if id.startswith('v'):
                    self.table_data[target_key][tagw] = f"{self.clk}"
                else:
                    self.table_data[target_key][tagw] = f"{self.clk} {current_content.split()[-1] if current_content else id}"

            if self.stations_to_write[id]['write']:
                reg_dest = self.stations_to_write[id]['regs'][0]
                val_fwd = self.stations_to_write[id]['fwd']
                estado_atual = self.registers.get(reg_dest, 0.0)
                if isinstance(estado_atual, tuple):
                    if estado_atual[0] == id and len(estado_atual) == 3 and estado_atual[2] == self.stations_to_write[id]['ticket']:
                        self.registers[reg_dest] = val_fwd
                    else: self.registers[reg_dest] = (estado_atual[0], val_fwd, estado_atual[2])
                else: self.registers[reg_dest] = val_fwd
            elif self.stations_to_write[id]['inst'] and self.stations_to_write[id]['inst'].startswith('sd'):
                val_to_write = self.stations_to_write[id]['regs'][0]
                endereco_memoria = min(max(0, self.stations_to_write[id]['regs'][1] + self.stations_to_write[id]['regs'][2]), len(self.memory) - 1)
                self.memory[endereco_memoria] = val_to_write

        self.prev_fwd = {sid: sw['fwd'] for sid, sw in self.stations_to_write.items() if sw['fwd'] is not None}
        self.stations_to_write = {}

        # FORWARD
        for id in self.stations:
            if self.stations[id]['end'] is None or self.stations[id]['end'] < self.clk: continue
            tagf = 'execute_1' if self.stations[id]['branch'] else 'execute'
            regs = self.stations[id]['regs']

            if self.stations[id]['end'] == self.clk:
                inst = self.stations[id]['inst']
                if inst.startswith('bne'):
                    reg1_val, reg2_val = self.stations[id]['regs'][1], self.stations[id]['regs'][2]
                    if not self.stop:
                        if reg1_val != reg2_val:
                            self.instructions = self.branch_instructions.copy()
                            self.done_branch = True
                        else: self.stop = True
                    self.stations_to_write[id] = self.stations[id].copy()
                    self.stations_to_reset.append(id)
                    continue

                target_key = inst
                for k in self.table_data:
                    if k.startswith(inst): target_key = k; break

                if not id.startswith('i'): self.table_data[target_key][tagf] += f'{self.clk}'

                fwd = None
                if self.stations[id]['write']:
                    r1 = self.stations[regs[1]]['fwd'] if (isinstance(regs[1], str) and regs[1] in self.stations and self.stations[regs[1]]['fwd'] is not None) else self.prev_fwd.get(regs[1], regs[1])
                    r2 = self.stations[regs[2]]['fwd'] if (isinstance(regs[2], str) and regs[2] in self.stations and self.stations[regs[2]]['fwd'] is not None) else self.prev_fwd.get(regs[2], regs[2])
                    self.stations[id]['regs'][1], self.stations[id]['regs'][2] = r1, r2

                    if id[0] == 'l': fwd = self.memory[min(max(0, r1 + r2), len(self.memory) - 1)]
                    elif id[0] == 'm': fwd = r1 * r2
                    elif id[0] == 'i': fwd = r1 - r2 if self.stations[id]['inst'].split()[0] in ['sub', 'subi'] else r1 + r2
                    elif id[0] == 'v': fwd = r1
                    else: fwd = r1 + r2

                self.stations[id]['fwd'] = fwd
                self.stations_to_write[id] = self.stations[id].copy()
                self.stations_to_reset.append(id)

        # EXECUTE
        for id in self.stations:
            if self.stations[id]['end'] is None or self.stations[id]['end'] < self.clk: continue
            tage = 'execute_1' if self.stations[id]['branch'] else 'execute'
            tagw = 'write_1' if self.stations[id]['branch'] else 'write'
            inst = self.stations[id]['inst']

            if self.clk == (self.stations[id]['end'] - self.cicles.get(id[0], 1) + 1 if not id.startswith('i') else self.stations[id]['end']):
                target_key = inst
                for k in self.table_data:
                    if k.startswith(inst): target_key = k; break
                self.table_data[target_key][tage] = f'{self.clk}'

                if id.startswith('v'):
                    self.table_data[target_key][tagw] = ''
                else:
                    self.table_data[target_key][tagw] = f'{id}'

            if not id.startswith('i') and not id.startswith('v') and self.stations[id]['end'] - self.cicles[id[0]] + 1 == self.clk:
                target_key = inst
                for k in self.table_data:
                    if k.startswith(inst): target_key = k; break
                if self.table_data[target_key][tage] == f'{self.clk}': self.table_data[target_key][tage] += '-'
                else: self.table_data[target_key][tage] += f'..{self.clk}-'

            for r_idx in [1, 2]:
                r_val = self.stations[id]['regs'][r_idx]
                if isinstance(r_val, str) and r_val in self.stations:
                    self.stations[id]['regs'][r_idx] = self.stations[r_val]['fwd'] if self.stations[r_val]['fwd'] is not None else self.prev_fwd.get(r_val, r_val)

        for id in self.stations_to_reset: self.reset_station(id)
        self.stations_to_reset = []

        # DECODE
        tagd_d = 'decode_1' if self.done_branch else 'decode'
        if self.pipeline['decode']:
            inst = self.pipeline['decode']
            clean_inst = inst.split('   #')[0].strip()

            if ":" in clean_inst:
                clean_inst = clean_inst.split(":", 1)[1].strip()

            if clean_inst.startswith('bne') and not self.stop:
                regs_bne = clean_inst.replace('bne', '').strip().split(',')
                station_id = self.get_station('bne', self.clk)
                if station_id:
                    if not self.done_branch:
                        self.instructions = self.branch_instructions.copy()
                        self.done_branch = True
                    else: self.stop = True
                    self.stations[station_id]['inst'] = clean_inst
                    self.stations[station_id]['write'] = False
                    self.stations[station_id]['regs'] = [None, self.parse_op(regs_bne[0].strip()), self.parse_op(regs_bne[1].strip())]
                    self.stations[station_id]['end'] = self.clk + self.cicles['i']
                    self.table_data[inst][tagd_d] = str(self.clk)
                    self.pipeline['decode'] = None
                    self.stall = (None, None)
            elif clean_inst.startswith('mov'):
                regs_mov = clean_inst.replace('mov', '').strip().split(',')
                dest_reg, src_reg = regs_mov[0].strip(), regs_mov[1].strip()

                self.mov_counter += 1
                station_id = f"v{self.mov_counter}"
                self.stations[station_id] = {
                    'inst': None, 'regs': [None] * 3, 'end': None,
                    'fwd': None, 'write': None, 'branch': False, 'ticket': None
                }
                if self.done_branch: self.stations[station_id]['branch'] = True
                self.stations[station_id]['inst'] = clean_inst
                self.check_dependencies_mov(station_id, dest_reg, src_reg)

                if self.table_data[inst][tagd_d] and self.table_data[inst][tagd_d].endswith('-'): self.table_data[inst][tagd_d] += str(self.clk)
                else: self.table_data[inst][tagd_d] = f"{self.table_data[inst][tagd_d]}, {self.clk}" if self.table_data[inst][tagd_d] else str(self.clk)
                self.pipeline['decode'] = None
                self.stall = (None, None)
            else:
                splitted = clean_inst.split(maxsplit=1)
                op = splitted[0]
                if clean_inst.startswith('ld') or clean_inst.startswith('sd'):
                    reg1, rest = splitted[1].replace(" ", "").split(',', 1)
                    offset, reg2 = rest.split('(', 1)
                    regs = [reg1, offset, reg2.rstrip(')')]
                else: regs = splitted[1].replace(" ", "").split(',')

                station_id = self.get_station(op, self.clk)
                if station_id:
                    if self.done_branch: self.stations[station_id]['branch'] = True
                    self.stations[station_id]['inst'] = clean_inst
                    self.check_dependencies(station_id, op, regs)
                    if self.table_data[inst][tagd_d] and self.table_data[inst][tagd_d].endswith('-'): self.table_data[inst][tagd_d] += str(self.clk)
                    else: self.table_data[inst][tagd_d] = f"{self.table_name if 'table_name' in dir() else self.table_data[inst][tagd_d]}, {self.clk}" if self.table_data[inst][tagd_d] else str(self.clk)
                    self.pipeline['decode'] = None
                    self.stall = (None, None)
                else:
                    self.stall = (self.next_available_station(op), clean_inst)
                    if not self.table_data[inst][tagd_d]: self.table_data[inst][tagd_d] = f"{self.clk}-"
                    else: self.table_data[inst][tagd_d] += f", {self.clk}-"

        # FETCH
        tagf = 'fetch_1' if self.done_branch else 'fetch'
        if self.pipeline.get('fetch') is not None and self.pipeline.get('decode') is None:
            inst_in_fetch = self.pipeline['fetch']
            self.pipeline['decode'] = inst_in_fetch
            self.pipeline['fetch'] = None
            if str(self.table_data[inst_in_fetch][tagf]) != str(self.clk): self.table_data[inst_in_fetch][tagf] = f"{self.table_data[inst_in_fetch][tagf]}-{self.clk}"
        elif self.pipeline.get('fetch') is None and len(self.instructions) > 0:
            inst = self.instructions.pop(0)
            self.pipeline['fetch'] = inst
            self.table_data[inst][tagf] = str(self.clk)
            if self.pipeline.get('decode') is None:
                self.pipeline['decode'] = inst; self.pipeline['fetch'] = None

        all_finished = True
        for k, stages in self.table_data.items():
            clean_name = k.split('   #')[0].strip()
            if clean_name.startswith('bne'):
                continue
            if clean_name.startswith('sd'):
                exe_val = str(stages.get('execute_1', "")).strip()
                if not (exe_val and '-' in exe_val):
                    all_finished = False
                    break
            else:
                w_val = str(stages.get('write_1', "")).strip()
                if not any(n.isdigit() for n in w_val.split()):
                    all_finished = False
                    break
        if all_finished and len(self.table_data) > 0:
            self.forced_stop = True

### *Parsing Assembly* $\rightarrow$ *Python*

In [ ]:
# TABELA DE INSTRUCOES
INSTR_TABLE = {
    "mulf":  ("r_type", "*"),
    "multf": ("r_type", "*"),
    "mul":   ("r_type", "*"),
    "mult":  ("r_type", "*"), # Todos os muls viram mulf
    "addf":  ("r_type", "+"),
    "add":   ("r_type", "+"),
    "sub":   ("r_type", "-"),
    "addi":  ("i_type", "+"),
    "subi":  ("i_type", "-"),
    "ld":    ("load", None),
    "sd":    ("store", None),
    "bne":   ("branch", "!="),
    "beq":   ("branch", "=="),
    "j":     ("jump", None),
    "mov":   ("mov", None)
}



MEM_RE = re.compile(r"^(-?\d+)\((\w+)\)$")

def _reg_name(token):
    m = re.match(r"^([fr])(\d+)$", token)
    if m:
        bank, idx = m.groups()
        return f"{bank}[{idx}]"
    return token

def parse_line(raw_line, line_num=None):
    line = raw_line.split("#")[0].strip()
    if not line:
        return None

    # Aceita instruções, registradores e labels em UPPERCASE ou lowercase
    line = line.lower()

    label = None
    # Separação robusta de label (funciona com ou sem espaço após o ':')
    if ":" in line:
        parts = line.split(":", 1)
        possible_label = parts[0].strip()
        # Garante que não há espaços internos no nome da label
        if " " not in possible_label and possible_label:
            label = possible_label
            line = parts[1].strip()
            if not line:
                return {"type": "label", "label": label, "raw": raw_line.strip(),
                        "python": f"# label: {label}"}

    m = re.match(r"^(\S+)\s+(.*)$", line)
    if not m:
        raise ValueError(f"Linha {line_num}: não consegui interpretar '{raw_line.strip()}'")
    opcode, rest = m.groups()

    # 💥 CORREÇÃO: Extrai os operandos PRIMEIRO para que fiquem disponíveis na checagem abaixo
    operands = [op.strip() for op in rest.split(",")]

    # Normalização de opcodes para a identidade padrão da máquina (Tomasulo friendly)
    if opcode in ["mulf", "multf", "mul", "mult"]:
        opcode = "mulf"
    elif opcode == "add" and any('f' in op for op in operands):
        opcode = "addf"

    if opcode not in INSTR_TABLE:
        raise ValueError(f"instrução desconhecida '{opcode}'")

    category, op = INSTR_TABLE[opcode]
    result = {"opcode": opcode, "category": category, "op": op, "label": label,
              "raw": raw_line.strip(), "operands": operands}

    expected_operands = {
        "r_type": 3, "i_type": 3, "load": 2, "store": 2, "branch": 3, "jump": 1, "mov": 2
    }
    if len(operands) != expected_operands[category]:
        raise ValueError(f"'{opcode}' espera {expected_operands[category]} operando(s), recebeu {len(operands)}")

    if category == "r_type":
        dest, src1, src2 = operands
        result["python"] = f"{_reg_name(dest)} = {_reg_name(src1)} {op} {_reg_name(src2)}"
    elif category == "i_type":
        dest, src, imm = operands
        result["python"] = f"{_reg_name(dest)} = {_reg_name(src)} {op} {imm}"
    elif category == "load":
        dest, mem = operands
        mm = MEM_RE.match(mem)
        if not mm: raise ValueError(f"endereço de memória inválido '{mem}'")
        offset, base = mm.groups()
        result["python"] = f"{_reg_name(dest)} = mem[{_reg_name(base)} + {offset}]"
    elif category == "store":
        src, mem = operands
        mm = MEM_RE.match(mem)
        if not mm: raise ValueError(f"endereço de memória inválido '{mem}'")
        offset, base = mm.groups()
        result["python"] = f"mem[{_reg_name(base)} + {offset}] = {_reg_name(src)}"
    elif category == "mov":
        dest, src = operands
        result["python"] = f"{_reg_name(dest)} = {_reg_name(src)}"
    elif category == "branch":
        src1, src2, target = operands
        result["python"] = f"if {_reg_name(src1)} {op} {_reg_name(src2)}: goto('{target}')"
    elif category == "jump":
        (target,) = operands
        result["python"] = f"goto('{target}')"

    if label:
        result["python"] = f"# {label}:\n" + result["python"]
    return result

def extract_dest_reg(text):
    """Extrai o registrador de destino de uma instrução se ele for de ponto flutuante (fX)"""
    text = text.split('#')[0].strip().lower()
    parts = text.split(None, 1)
    if len(parts) < 2: return None
    opcode, rest = parts
    # Ignora instruções que não escrevem em registrador de destino flutuante
    if opcode in ['sd', 'sw', 'sb', 'sh', 'bne', 'beq', 'bgt', 'blt', 'j']:
        return None
    subparts = rest.split(',')
    if subparts:
        possible_reg = subparts[0].strip()
        if re.match(r'^f\d+$', possible_reg):
            return possible_reg
    return None

def parse_program(text):
    out = []
    for i, raw_line in enumerate(text.split("\n"), 1):
        parsed = parse_line(raw_line, line_num=i)
        if parsed:
            out.append(parsed)
    return out

---

## Ferramenta interativa

In [ ]:
#@title Editor de Instruções
custom_css = widgets.HTML("""
<style>
.header-card {
    background: linear-gradient(135deg, #1d3557 0%, #457b9d 100%);
    padding: 18px 24px; border-radius: 14px 14px 0 0; color: white; font-family: 'Segoe UI', sans-serif;
}
.header-card h3 { margin: 0; font-size: 20px; font-weight: 600; }
.header-card p { margin: 4px 0 0 0; font-size: 13px; opacity: 0.85; }
.tool-card { border: 1px solid #dcdde1; border-radius: 14px; padding: 0 0 18px 0; background: #fafbfc; box-shadow: 0 4px 16px rgba(0,0,0,0.08); margin-bottom: 16px; }
.inner-content { padding: 18px 24px 0 24px; }
.widget-textarea textarea { border-radius: 10px !important; border: 1.5px solid #a8dadc !important; font-family: 'Consolas','Courier New',monospace !important; font-size: 20px !important; line-height: 1.5 !important; padding: 12px !important; }
.widget-button { border-radius: 8px !important; font-weight: 600 !important; font-size: 13px !important; height: 38px !important; }
.output-box { background: white; border-radius: 10px; border: 1px solid #e0e0e0; padding: 8px 14px; font-family: 'Consolas','Courier New',monospace; font-size: 15px; color: #1d1d1d !important; }
.output-box pre { color: #1d1d1d !important; background: transparent !important; }
.output-box * { color: #1d1d1d !important; }
.config-panel { background: #eef2f7; border: 1px solid #dcdde1; border-radius: 10px; padding: 14px 20px; margin: 10px 0; font-family: 'Segoe UI', sans-serif; }
</style>
""")

# ---- estado do programa e da máquina fixa (compartilhados entre as células) ----
program_instructions = []
machine_config = MachineConfig(mem_size=200, word_size=4)

# ---------------- Widgets de entrada do programa ----------------
input_area = widgets.Textarea(
    value=(
        'loop:ld f3,0(r1)\n'
        'multf f1,f2,f3\n'
        'ld f4,4(r1)\n'
        'mulf f5,f1,f4\n'
        'mult f4,f4,f2\n'
        'sd f5,0(r1)\n'
        'addf f5,f5,f4\n'
        'sd f5,4(r1)\n'
        'addi r1,r1,8\n'
        'bne r1,r2,loop'
    ),
    description='',
    layout=widgets.Layout(width='95%', height='320px', margin='0 0 12px 0')
)

add_button = widgets.Button(description='Adicionar ao programa', icon='plus', button_style='success', layout=widgets.Layout(width='220px'))
clear_button = widgets.Button(description='Limpar programa', icon='trash', button_style='danger', layout=widgets.Layout(width='180px'))
graph_button = widgets.Button(description='Gerar grafo de dependências', icon='project-diagram', button_style='info', layout=widgets.Layout(width='260px'))

output = widgets.Output(layout=widgets.Layout(margin='14px 0 0 0', max_height='300px', overflow='auto'))
graph_output = widgets.Output(layout=widgets.Layout(margin='10px 0 0 0'))

def on_add_clicked(b):
    with output:
        clear_output()
        try:
            novas = parse_program(input_area.value)
            program_instructions.extend(novas)
            print(f"✅ {len(novas)} instrução(ões) adicionada(s). Total no programa: {len(program_instructions)}\n")
            for i, instr in enumerate(program_instructions):
                print(f"[{i}] {instr['raw']}  ->  {instr.get('python', '<<< SEM TRADUÇÃO >>>')}")
        except ValueError as e:
            print(f"❌ Erro de parsing: {e}")

def on_clear_clicked(b):
    global program_instructions
    program_instructions = []
    input_area.value = ''
    with output:
        clear_output(); print("🗑️ Programa limpo.")
    with graph_output:
        clear_output()

def build_dependency_graph(instructions):
    G = nx.DiGraph()
    for i, instr in enumerate(instructions):
        if instr.get('type') == 'label':
            continue

        opcode = instr['opcode']
        if opcode in ["mulf", "multf", "mul", "mult"]:
            opcode = "mult"
        elif opcode in ["addf", "add"]:
            opcode = "add"

        display_text = f"{opcode} {', '.join(instr['operands'])}"
        G.add_node(i, text=display_text)

    last_write = {}
    edges = defaultdict(set)

    for j, instr in enumerate(instructions):
        if instr.get('type') == 'label' or instr['category'] in ('branch', 'jump'):
            continue

        opcode = instr['opcode']
        operands = instr['operands']
        find_f = lambda t: set(re.findall(r'f\d+', t.lower()))

        writes_j, reads_j = set(), set()

        if opcode in ('mult', 'mul', 'mulf', 'multf'):
            writes_j |= find_f(operands[0])
            if len(operands) > 1: reads_j |= find_f(operands[1])
            if len(operands) > 2: reads_j |= find_f(operands[2])
        elif opcode in ('add', 'addf'):
            writes_j |= find_f(operands[0])
            if len(operands) > 1: reads_j |= find_f(operands[1])
            if len(operands) > 2: reads_j |= find_f(operands[2])
        elif opcode == 'ld':
            writes_j |= find_f(operands[0])
            if len(operands) > 1: reads_j |= find_f(operands[1])
        elif opcode == 'sd':
            reads_j |= find_f(operands[0])
            if len(operands) > 1: reads_j |= find_f(operands[1])
        elif opcode == 'mov':
            writes_j |= find_f(operands[0])
            if len(operands) > 1: reads_j |= find_f(operands[1])

        for r in reads_j:
            if r in last_write:
                edges[(last_write[r], j)].add(r)

        for r in writes_j:
            last_write[r] = j

    for (u, v), regs in edges.items():
        G.add_edge(u, v, regs=regs)
    return G

def compute_layered_positions(G):
    level = {}
    for node in nx.topological_sort(G):
        preds = list(G.predecessors(node))
        level[node] = 0 if not preds else max(level[p] for p in preds) + 1
    return level

def on_graph_clicked(b):
    with graph_output:
        clear_output()
        if not program_instructions:
            print("⚠️ Nenhuma instrução no programa ainda. Adicione instruções primeiro.")
            return
        G = build_dependency_graph(program_instructions)
        no_dep = [n for n in G.nodes() if G.degree(n) == 0]
        G.remove_nodes_from(no_dep)
        if G.number_of_nodes() == 0:
            print("ℹ️ Nenhuma instrução com dependência de registradores 'f' para exibir.")
            return

        level = compute_layered_positions(G)
        levels = defaultdict(list)
        for node, lvl in level.items():
            levels[lvl].append(node)

        dot = graphviz.Digraph(comment='Grafo de Dependencia Puro RAW', format='svg')
        dot.attr(rankdir='TB', splines='true', nodesep='1.1', ranksep='0.7')

        dot.attr('node', fontname='monospace', shape='ellipse', style='filled',
                 fillcolor='#f1faee', color='#1d3557', penwidth='2.2', fontcolor='#1d3557', fontsize='12')
        dot.attr('edge', fontname='sans-serif', color='#e63946', penwidth='1.8', fontsize='11', fontcolor='#e63946', weight='1.2')

        max_level = max(level.values())

        for lvl in range(max_level + 1):
            dot.node(f'L_{lvl}', f'N = {lvl}', shape='plaintext', fillcolor='none',
                     color='none', fontcolor='#2d3436', fontsize='13', fontweight='bold')
            if lvl > 0:
                dot.edge(f'L_{lvl-1}', f'L_{lvl}', style='invis')

        for lvl in sorted(levels.keys()):
            with dot.subgraph() as s:
                s.attr(rank='same')
                s.node(f'L_{lvl}')
                for node in levels[lvl]:
                    s.node(str(node), G.nodes[node]['text'])

        for u, v, data in G.edges(data=True):
            reg_label = ", ".join(sorted(data['regs']))
            dot.edge(str(u), str(v), label=f" {reg_label} ")

        display(dot)

add_button.on_click(on_add_clicked)
clear_button.on_click(on_clear_clicked)
graph_button.on_click(on_graph_clicked)

output.add_class('output-box')
graph_output.add_class('output-box')

header = widgets.HTML("""
<div class="header-card">
    <h3>🔧 Editor de Instruções — Programa + Configuração da Máquina</h3>
    <p>Digite instruções Assembly, adicione ao programa e gere o grafo de dependências.</p>
</div>
""")

config_panel = widgets.HTML(f"""
<div class="config-panel">
    <span style="color: #1d3557; font-weight: bold; font-size: 14px;">⚙️ Configuração Fixa da Máquina:</span><br/>
    <span style="font-size: 13px; color: #2d3436;">
        • <b>Tamanho da Memória:</b> {machine_config.mem_size} posições |
        • <b>Word Size:</b> {machine_config.word_size} bytes<br/>
        • <b>Estado Inicial dos Registradores:</b> <code style="background: #ffffff; padding: 2px 4px; border-radius: 4px;">r1=1, r2=2, ..., r31=31</code> e <code style="background: #ffffff; padding: 2px 4px; border-radius: 4px;">f1=1.0, f2=2.0, ..., f31=31.0</code>
    </span>
</div>
""")

body = widgets.VBox([
    input_area,
    widgets.HBox([add_button, clear_button, graph_button], layout=widgets.Layout(gap='10px')),
    config_panel,
    output,
    graph_output
], layout=widgets.Layout())
body.add_class('inner-content')

card = widgets.VBox([header, body])
card.add_class('tool-card')

display(custom_css, card)

HTML(value="\n<style>\n.header-card {\n    background: linear-gradient(135deg, #1d3557 0%, #457b9d 100%);\n   …

### Gera grafo de soft. pipeline e compara com original

In [ ]:
#@title Usuário propõe grafo e possui gabarito { display-mode: "form" }

# =========================================================================
# 🧬 ESTILOS CSS - RESPONSIVIDADE E CORREÇÃO DAS ABAS (DARK MODE)
# =========================================================================
css_unificado = widgets.HTML("""
    <style>
    .no-pointer-plots canvas, .no-pointer-plots .jupyter-matplotlib { pointer-events: none !important; }
    /* Mantém os grafos de ambas as abas 100% responsivos */
    .responsive-graph svg, .responsive-anim-graph svg { width: 100% !important; height: auto !important; max-width: 100% !important; }

    /* 🚀 CORREÇÃO DEFINITIVA DAS ABAS DO IPYWIDGETS NO COLAB */
    .lm-TabBar-tab, .p-TabBar-tab {
        width: auto !important;             /* Alarga a aba baseado no tamanho do texto */
        min-width: 220px !important;        /* Define uma largura mínima segura alargada */
        padding: 0 20px !important;         /* Dá um espaçamento interno confortável */
        overflow: visible !important;       /* Impede o corte de letras */
    }
    .lm-TabBar-tabLabel, .p-TabBar-tabLabel {
        color: #ffffff !important;          /* Força o texto a ficar 100% BRANCO */
        font-weight: 600 !important;        /* Deixa o texto em negrito para destacar */
        font-size: 13px !important;
        overflow: visible !important;
        text-overflow: clip !important;
    }
    /* Altera a cor de fundo das abas não selecionadas para dar contraste */
    .lm-TabBar-tab:not(.lm-mod-current), .p-TabBar-tab:not(.p-mod-current) {
        background-color: #334155 !important;
        opacity: 0.7;
    }
    /* Destaca a aba que está ativa no momento */
    .lm-TabBar-tab.lm-mod-current, .p-TabBar-tab.p-mod-current {
        background-color: #1e293b !important;
        border-bottom: 2px solid #38bdf8 !important;
        opacity: 1;
    }
    </style>
    <div style="background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%); padding: 18px 24px; border-radius: 12px 14px 0 0; color: white; font-family: sans-serif;">
        <h3 style="margin: 0; font-size: 22px; font-weight: 600;">🛠️ Central de Grafos: Pipeline de Software</h3>
        <p style="margin: 4px 0 0 0; font-size: 13px; opacity: 0.85;">Proponha o seu grafo no Canvas ou navegue pelo Gabarito Evolutivo gerado automaticamente.</p>
    </div>
""")

# =========================================================================
# ✏️ CONFIGURAÇÃO DE WIDGETS - ABA 1: CANVAS DO USUÁRIO
# =========================================================================
txt_node_instruction = widgets.Text(value='mov f4, f1', placeholder='Ex: mult f2, f1, f1', description='Instrução:', layout=widgets.Layout(width='280px'))
slider_node_level = widgets.IntSlider(value=0, min=0, max=8, step=1, description='Nível (N):', layout=widgets.Layout(width='240px'))
btn_add_canvas_node = widgets.Button(description='Adicionar Nó', icon='plus', button_style='success', layout=widgets.Layout(width='140px', height='34px'))

txt_graph_code = widgets.Textarea(
    value='# Formato esperado: [nivel] inst\n[0] ld f3, 0(r1)\n[0] ld f4, 4(r1)\n[1] mult f1, f2, f3\n[1] mov f6, f4\n[1] mult f7, f4, f2\n[2] mult f5, f1, f6\n[2] mov f8, f7\n[3] sd f5, 0(r1)\n[3] add f9, f5, f8\n[4] sd f9, 4(r1)',
    placeholder='Digite o grafo por extenso...',
    description='Script Grafo:',
    layout=widgets.Layout(width='675px', height='125px', margin='5px 0 10px 0')
)
btn_load_graph_code = widgets.Button(description='Carregar Grafo por Escrito', icon='code', button_style='info', layout=widgets.Layout(width='230px', height='34px', margin='40px 0 0 0'))

drop_remove_node = widgets.Dropdown(options=[('Nenhum nó disponível', -1)], description='Selecionar Nó:', style={'description_width': 'initial'}, layout=widgets.Layout(width='320px'), disabled=True)
btn_remove_canvas_node = widgets.Button(description='Remover Selecionado', icon='trash', button_style='warning', layout=widgets.Layout(width='180px', height='34px'))
btn_clear_canvas = widgets.Button(description='Limpar Canvas', icon='trash-restore', button_style='danger', layout=widgets.Layout(width='140px', height='34px'))
btn_validate_canvas_graph = widgets.Button(description='Validar Grafo Proposto', icon='shield-check', button_style='primary', layout=widgets.Layout(width='220px', height='34px'))

canvas_output_plot = widgets.Output(layout=widgets.Layout(margin='14px 0 0 0'))
user_canvas_nodes = []

# =========================================================================
# 📖 CONFIGURAÇÃO DE WIDGETS - ABA 2: GABARITO PASSO A PASSO
# =========================================================================
btn_graph_prev = widgets.Button(description='Passo Anterior', icon='arrow-left', button_style='warning', layout=widgets.Layout(width='160px', height='40px'))
btn_graph_next = widgets.Button(description='Próximo Passo', icon='arrow-right', button_style='success', layout=widgets.Layout(width='160px', height='40px'))
btn_graph_final = widgets.Button(description='Ir para o Final', icon='fast-forward', button_style='info', layout=widgets.Layout(width='160px', height='40px'))

lbl_graph_step = widgets.Label(value='Passo 0 de 0', layout=widgets.Layout(margin='8px 0 0 15px'))
lbl_graph_step.style.text_color = '#f1f5f9'
pipe_compare_output = widgets.Output(layout=widgets.Layout(margin='14px 0 0 0'))

current_graph_idx = 0
pipeline_cache = {}

# =========================================================================
# 🧮 FUNÇÕES DE SUPORTE DO CANVAS (ABA 1)
# =========================================================================
def update_remove_dropdown():
    if not user_canvas_nodes:
        drop_remove_node.options = [('Nenhum nó disponível', -1)]; drop_remove_node.disabled = True
    else:
        drop_remove_node.options = [(f"[{idx}] {item['text']} (Nível {item['level']})", idx) for idx, item in enumerate(user_canvas_nodes)]
        drop_remove_node.disabled = False

def build_user_proposed_graph(nodes_list):
    G = nx.DiGraph()
    for idx, item in enumerate(nodes_list):
        parsed = parse_line(item['text'], 0)
        opcode = parsed['opcode'] if parsed else 'mov'
        if opcode in ["mulf", "multf", "mul", "mult"]: opcode = "mult"
        elif opcode in ["addf", "add"]: opcode = "add"
        cleaned_text = f"{opcode} {', '.join(parsed['operands'])}" if parsed else item['text']
        G.add_node(idx, text=cleaned_text, level=item['level'])

    last_write = {}
    edges = defaultdict(set)
    for j, item in enumerate(nodes_list):
        parsed = parse_line(item['text'], 0)
        if not parsed: continue
        opcode = parsed['opcode']
        if opcode in ["mulf", "multf", "mul", "mult"]: opcode = "mult"
        elif opcode in ["addf", "add"]: opcode = "add"
        operands = parsed['operands']
        find_f = lambda t: set(re.findall(r'f\d+', t.lower()))

        writes_j, reads_j = set(), set()
        if opcode in ('mult', 'add', 'sub'):
            writes_j |= find_f(operands[0]); reads_j |= find_f(operands[1])
            if len(operands) > 2: reads_j |= find_f(operands[2])
        elif opcode == 'ld': writes_j |= find_f(operands[0])
        elif opcode == 'sd': reads_j |= find_f(operands[0])
        elif opcode == 'mov': writes_j |= find_f(operands[0]); reads_j |= find_f(operands[1])

        for r in reads_j:
            if r in last_write: edges[(last_write[r], j)].add(r)
        for r in writes_j:
            last_write[r] = j

    for (u, v), regs in edges.items(): G.add_edge(u, v, regs=regs)
    return G

def make_html_label_local(text, dest_changed=False):
    text = text.lower().strip()
    parts = text.split(None, 1)
    if len(parts) == 2:
        opcode, rest = parts; subparts = rest.split(',', 1)
        if len(subparts) == 2:
            dest, remaining = subparts
            color = "red" if dest_changed else "#1d3557"
            return f'<<table border="0" cellborder="0" cellspacing="0"><tr><td><font color="#1d3557"><b>{opcode} </b></font></td><td><font color="{color}"><b>{dest.strip()}</b></font></td><td><font color="#1d3557"><b>, {remaining.strip()}</b></font></td></tr></table>>'
    return f'<<table border="0" cellborder="0" cellspacing="0"><tr><td><font color="#1d3557"><b>{text}</b></font></td></tr></table>>'

def generate_graphviz_panel(G, level_dict, node_text_map, dest_changed_map=None):
    dot = graphviz.Digraph(format='svg')
    dot.attr(rankdir='TB', splines='true', nodesep='1.0', ranksep='0.6')
    dot.attr('node', fontname='monospace', shape='ellipse', style='filled', fillcolor='white', color='#1e293b', penwidth='2.0', fontcolor='#1e293b', fontsize='12')
    dot.attr('edge', fontname='sans-serif', color='#475569', penwidth='1.8', fontsize='11', fontcolor='#475569')

    if not level_dict: return dot
    max_level = max(level_dict.values())
    for lvl in range(max_level + 1):
        dot.node(f'L_{lvl}', f'N = {lvl}', shape='plaintext', fillcolor='none', color='none', fontcolor='#475569', fontsize='13', fontweight='bold')
        if lvl > 0: dot.edge(f'L_{lvl-1}', f'L_{lvl}', style='invis')

    levels_to_nodes = defaultdict(list)
    for n, lvl in level_dict.items(): levels_to_nodes[lvl].append(n)

    for lvl in sorted(levels_to_nodes.keys()):
        with dot.subgraph() as s:
            s.attr(rank='same'); s.node(f'L_{lvl}')
            for node in levels_to_nodes[lvl]:
                is_changed = dest_changed_map[node] if dest_changed_map else False
                lbl = make_html_label_local(node_text_map[node], is_changed)
                s.node(str(node), lbl)

    for u, v, data in G.edges(data=True):
        dot.edge(str(u), str(v), label=f" {', '.join(sorted(data['regs']))} ")
    return dot

def render_canvas_split_view():
    with canvas_output_plot:
        clear_output(wait=True)
        G_orig = build_dependency_graph(program_instructions)
        no_dep = [n for n in G_orig.nodes() if G_orig.degree(n) == 0]; G_orig.remove_nodes_from(no_dep)

        if G_orig.number_of_nodes() == 0:
            print("⚠️ Aguardando carregamento de instruções válidas na Célula 2.")
            return

        level_orig = compute_layered_positions(G_orig)
        node_text_map_orig = {n: G_orig.nodes[n]['text'] for n in G_orig.nodes()}
        dot_orig = generate_graphviz_panel(G_orig, level_orig, node_text_map_orig)

        original_written = ISASimulator.collect_written_regs(program_instructions)
        G_user = build_user_proposed_graph(user_canvas_nodes)
        level_user = {idx: item['level'] for idx, item in enumerate(user_canvas_nodes)}
        node_text_map_user = {idx: item['text'] for idx, item in enumerate(user_canvas_nodes)}

        def is_renamed(text):
            parts = text.split(None, 1)
            if len(parts) < 2: return False
            opcode, rest = parts; dest = rest.split(',')[0].strip()
            return opcode == 'mov' or dest not in original_written

        dest_changed_user = {idx: is_renamed(item['text']) for idx, item in enumerate(user_canvas_nodes)}
        dot_user = generate_graphviz_panel(G_user, level_user, node_text_map_user, dest_changed_user)

        out_left, out_right = widgets.Output(), widgets.Output()
        out_left.add_class('responsive-graph')
        out_right.add_class('responsive-graph')

        with out_left: display(dot_orig)
        with out_right:
            if user_canvas_nodes: display(dot_user)
            else: print("\n\n\n\n[ Canvas em Branco ]")

        # ALTERAÇÃO: Mantém os títulos aumentados e o alinhamento perfeito também na Aba 1
        title_style = 'color: #ffffff; text-align: center; font-weight: bold; font-family: sans-serif; font-size: 18px; margin: 0 0 15px 0; padding: 0;'

        split_panel = widgets.HBox([
            widgets.VBox([widgets.HTML(f'<h3 style="{title_style}">1. Grafo Original (Referência)</h3>'), out_left], layout=widgets.Layout(width='50%')),
            widgets.VBox([widgets.HTML(f'<h3 style="{title_style}">2. Seu Grafo Proposto</h3>'), out_right], layout=widgets.Layout(width='50%'))
        ], layout=widgets.Layout(width='100%', align_items='flex-start'))
        display(split_panel)

def on_validate_canvas_clicked(b):
    render_canvas_split_view()
    with canvas_output_plot:
        # Trava de Segurança Contra Canvas Vazio
        if not user_canvas_nodes:
            print("\n❌ ERRO DE VALIDAÇÃO: O canvas está completamente vazio!")
            print("Insira instruções nas camadas ou use o 'Script Grafo' antes de submeter para homologação.\n")
            return

        validator = PipelineValidator(program_instructions, machine_config)
        result = validator.validate(user_canvas_nodes)
        if result.get("error"):
            print(f"❌ {result['error']}"); return
        n_show = min(30, machine_config.mem_size)
        if result["success"]:
            print("\n✅ VEREDICTO: SEU GRAFO DE PIPELINE ESTÁ CORRETO!")
            print("A distribuição de níveis e as dependências de registradores geraram equivalência semântica perfeita na memória.\n")
        else:
            print("\n❌ VEREDICTO: GRAFO INCORRETO (Hazard ou Renomeação Inválida)")
        print(f"    Gabarito Esperado : {result['mem_gabarito'][:n_show]}")
        print(f"    Sua Saída         : {result['mem_pipe'][:n_show]}")

def on_add_node_clicked(b):
    inst_text = txt_node_instruction.value.strip()
    if inst_text:
        try:
            parsed_result = parse_line(inst_text.lower(), line_num=len(user_canvas_nodes) + 1)
            if parsed_result:
                user_canvas_nodes.append({'text': inst_text.lower(), 'level': slider_node_level.value})
                update_remove_dropdown(); render_canvas_split_view()
        except ValueError as e:
            with canvas_output_plot: print(f"❌ REJEITADO PELO PARSER: {e}")

def on_load_graph_code_clicked(b):
    global user_canvas_nodes
    code_block = txt_graph_code.value.strip()
    if not code_block: return
    parsed_nodes = []
    try:
        for idx, line in enumerate(code_block.split('\n'), 1):
            line_clean = line.strip()
            if not line_clean or line_clean.startswith('#'): continue
            match = re.match(r"^\[\s*(\d+)\s*(?:,\s*\d+\s*)?\]\s*(.+)$", line_clean)
            if not match: raise ValueError(f"Linha {idx}: Formato de tags inválido.")
            lvl_str, inst_raw = match.groups(); lvl = int(lvl_str); inst_raw = inst_raw.strip().lower()
            if parse_line(inst_raw, line_num=idx): parsed_nodes.append({'text': inst_raw, 'level': lvl})
        user_canvas_nodes = parsed_nodes
        update_remove_dropdown(); render_canvas_split_view()
    except ValueError as e:
        with canvas_output_plot: print(f"❌ ERRO AO PARSEAR SCRIPT DO GRAFO: {e}")

def on_remove_node_clicked(b):
    idx_to_remove = drop_remove_node.value
    if idx_to_remove is not None and 0 <= idx_to_remove < len(user_canvas_nodes):
        user_canvas_nodes.pop(idx_to_remove)
        update_remove_dropdown(); render_canvas_split_view()

def on_clear_canvas_clicked(b):
    global user_canvas_nodes; user_canvas_nodes = []
    update_remove_dropdown(); render_canvas_split_view()

btn_add_canvas_node.on_click(on_add_node_clicked)
btn_load_graph_code.on_click(on_load_graph_code_clicked)
btn_remove_canvas_node.on_click(on_remove_node_clicked)
btn_clear_canvas.on_click(on_clear_canvas_clicked)
btn_validate_canvas_graph.on_click(on_validate_canvas_clicked)

# =========================================================================
# 🧮 FUNÇÕES DE SUPORTE DO GABARITO EVOLUTIVO (ABA 2)
# =========================================================================

def extract_dest_reg(text):
    text = text.split('#')[0].strip().lower()
    parts = text.split(None, 1)
    if len(parts) < 2: return None
    opcode, rest = parts
    if opcode in ['sd', 'sw', 'sb', 'sh', 'bne', 'beq', 'bgt', 'blt', 'j']:
        return None
    subparts = rest.split(',')
    if subparts:
        possible_reg = subparts[0].strip()
        if re.match(r'^f\d+$', possible_reg):
            return possible_reg
    return None

def build_pipelined_with_moves_ordered(G_orig, level_orig):
    all_regs = []
    for node in G_orig.nodes(): all_regs.extend(re.findall(r'f\d+', G_orig.nodes[node]['text']))
    reg_indices = [int(r[1:]) for r in all_regs if r.startswith('f')]
    global_next_reg = (max(reg_indices) + 1) if reg_indices else 1

    G_pipe = nx.DiGraph()
    level_pipe, node_text_map, dest_changed_map, nodes_order = {}, {}, {}, []
    levels_to_nodes = defaultdict(list)
    for node, lvl in level_orig.items(): levels_to_nodes[lvl].append(node)

    reg_name_at_level = defaultdict(dict)
    mov_nodes, move_node_counter, seen_destinations = {}, 1000, set()
    prod_consumers = defaultdict(list)

    for u, v, data in G_orig.edges(data=True):
        for r in data['regs']: prod_consumers[(u, r)].append(v)
    for u in G_orig.nodes():
        d = extract_dest_reg(G_orig.nodes[u]['text'])
        if d: reg_name_at_level[(u, d)][level_orig[u]] = d

    for lvl in sorted(levels_to_nodes.keys()):
        for (p, r), consumers in prod_consumers.items():
            L_p = level_orig[p]; L_end = max(level_orig[c] for c in consumers)
            if L_p < lvl <= L_end:
                if lvl < L_end:
                    mov_id = move_node_counter; move_node_counter += 1
                    r_prev = reg_name_at_level[(p, r)][lvl - 1]; r_new = f"f{global_next_reg}"; global_next_reg += 1
                    node_text_map[mov_id] = f"mov {r_new}, {r_prev}"
                    level_pipe[mov_id] = lvl; dest_changed_map[mov_id] = True
                    G_pipe.add_node(mov_id); nodes_order.append(mov_id)
                    mov_nodes[(p, r, lvl)] = mov_id; reg_name_at_level[(p, r)][lvl] = r_new
                    parent = p if lvl - 1 == L_p else mov_nodes[(p, r, lvl - 1)]
                    G_pipe.add_edge(parent, mov_id, regs={r_prev})
                else: reg_name_at_level[(p, r)][lvl] = reg_name_at_level[(p, r)][lvl - 1]

        for u in levels_to_nodes[lvl]:
            orig_text = G_orig.nodes[u]['text']
            src_mappings = {}
            for p_node in G_orig.predecessors(u):
                for r in G_orig[p_node][u]['regs']: src_mappings[r] = reg_name_at_level[(p_node, r)][lvl - 1]

            orig_dest = extract_dest_reg(orig_text); new_dest = orig_dest; dest_changed = False
            if orig_dest:
                if orig_dest in seen_destinations:
                    new_dest = f"f{global_next_reg}"; global_next_reg += 1; dest_changed = True
                seen_destinations.add(orig_dest)

            parts = orig_text.split(None, 1)
            if len(parts) == 2:
                opcode, rest = parts; subparts = rest.split(','); is_store = opcode.lower() in ['sd', 'sw', 'sb', 'sh']
                new_subparts = []
                for idx, sub in enumerate(subparts):
                    sub_clean = sub.strip()
                    if idx == 0 and not is_store and orig_dest is not None:
                        new_subparts.append(new_dest)
                    else:
                        updated_sub = sub_clean
                        for orig_src, mapped_name in src_mappings.items():
                            updated_sub = re.sub(r'\b' + re.escape(orig_src) + r'\b', mapped_name, updated_sub)
                        new_subparts.append(updated_sub)
                updated_text = f"{opcode} {', '.join(new_subparts)}"
            else: updated_text = orig_text

            node_text_map[u] = updated_text; level_pipe[u] = lvl; dest_changed_map[u] = dest_changed
            G_pipe.add_node(u); nodes_order.append(u)
            if orig_dest: reg_name_at_level[(u, orig_dest)][lvl] = new_dest
            for p_node in G_orig.predecessors(u):
                for r in G_orig[p_node][u]['regs']:
                    parent_in_pipe = p_node if lvl - 1 == level_orig[p_node] else mov_nodes[(p_node, r, lvl - 1)]
                    G_pipe.add_edge(parent_in_pipe, u, regs={reg_name_at_level[(p_node, r)][lvl - 1]})

    return G_pipe, level_pipe, node_text_map, dest_changed_map, nodes_order

def make_html_label_gabarito(text, dest_changed=False):
    text = text.lower().strip()
    parts = text.split(None, 1)
    if len(parts) == 2:
        opcode, rest = parts; subparts = rest.split(',', 1)
        if len(subparts) == 2:
            dest, remaining = subparts; color = "red" if dest_changed else "#1d3557"
            return f'<<table border="0" cellborder="0" cellspacing="0"><tr><td><font color="#1d3557"><b>{opcode} </b></font></td><td><font color="{color}"><b>{dest.strip()}</b></font></td><td><font color="#1d3557"><b>, {remaining.strip()}</b></font></td></tr></table>>'
    return f'<<table border="0" cellborder="0" cellspacing="0"><tr><td><font color="#1d3557"><b>{text}</b></font></td></tr></table>>'

def generate_graphviz_incremental(G, level_dict, node_text_map, dest_changed_map, visible_nodes_set, highlight_node=None, is_pipelined=False):
    dot = graphviz.Digraph(format='svg')
    dot.attr(rankdir='TB', splines='true', nodesep='1.0', ranksep='0.6')
    dot.attr('node', fontname='monospace', shape='ellipse', style='filled', fillcolor='white', color='#1e293b', penwidth='2.0', fontcolor='#1e293b', fontsize='12')
    dot.attr('edge', fontname='sans-serif', color='#475569', penwidth='1.8', fontsize='11', fontcolor='#475569')

    if not level_dict or not visible_nodes_set: return dot

    max_level = max(level_dict.values())
    for lvl in range(max_level + 1):
        dot.node(f'L_{lvl}', f'N = {lvl}', shape='plaintext', fillcolor='none', color='none', fontcolor='#475569', fontsize='13', fontweight='bold')
        if lvl > 0: dot.edge(f'L_{lvl-1}', f'L_{lvl}', style='invis')

    levels_to_nodes = defaultdict(list)
    for n in visible_nodes_set: levels_to_nodes[level_dict[n]].append(n)

    for lvl in sorted(levels_to_nodes.keys()):
        if lvl > max_level: break
        with dot.subgraph() as s:
            s.attr(rank='same'); s.node(f'L_{lvl}')
            for node in levels_to_nodes[lvl]:
                is_changed = dest_changed_map[node] if dest_changed_map else False
                lbl = make_html_label_gabarito(node_text_map[node], is_changed)
                if node == highlight_node and is_pipelined:
                    s.node(str(node), lbl, fillcolor='#dcfce7', color='#16a34a', penwidth='3.0')
                else:
                    s.node(str(node), lbl, fillcolor='white', color='#1e293b', penwidth='2.0')

    for u, v, data in G.edges(data=True):
        if u in visible_nodes_set and v in visible_nodes_set:
            reg_label = ", ".join(sorted(data['regs']))
            if v == highlight_node and is_pipelined:
                dot.edge(str(u), str(v), label=f" {reg_label} ", color='#2563eb', penwidth='2.2', fontcolor='#2563eb')
            else:
                dot.edge(str(u), str(v), label=f" {reg_label} ")
    return dot

def update_pipeline_cache():
    global pipeline_cache
    if 'program_instructions' not in globals() or not program_instructions:
        pipeline_cache = {}; return False
    if pipeline_cache.get('raw_program') == program_instructions: return True

    G_orig = build_dependency_graph(program_instructions)
    no_dep = [n for n in G_orig.nodes() if G_orig.degree(n) == 0]; G_orig.remove_nodes_from(no_dep)
    if G_orig.number_of_nodes() == 0: pipeline_cache = {}; return False

    level_orig = compute_layered_positions(G_orig)
    G_pipe, level_pipe, node_text_map_pipe, dest_changed_map_pipe, nodes_order = build_pipelined_with_moves_ordered(G_orig, level_orig)

    pipeline_cache = {
        'raw_program': list(program_instructions), 'G_orig': G_orig, 'level_orig': level_orig,
        'G_pipe': G_pipe, 'level_pipe': level_pipe, 'node_text_map_pipe': node_text_map_pipe,
        'dest_changed_map_pipe': dest_changed_map_pipe, 'nodes_order': nodes_order
    }
    return True

def render_current_graph_step():
    with pipe_compare_output:
        clear_output(wait=True)
        if not update_pipeline_cache():
            print("⚠️ Nenhuma instrução com dependência válida encontrada no programa."); return

        c = pipeline_cache; nodes_order = c['nodes_order']
        global current_graph_idx
        current_graph_idx = max(0, min(current_graph_idx, len(nodes_order) - 1))

        btn_graph_prev.disabled = (current_graph_idx == 0)
        btn_graph_next.disabled = (current_graph_idx == len(nodes_order) - 1)
        btn_graph_final.disabled = (current_graph_idx == len(nodes_order) - 1)

        visible_pipe_nodes = set(nodes_order[:current_graph_idx + 1])
        highlight_node = nodes_order[current_graph_idx]
        lbl_graph_step.value = f"Passo {current_graph_idx + 1} de {len(nodes_order)} | Nó: [{c['node_text_map_pipe'][highlight_node]}]"

        node_text_map_orig = {n: c['G_orig'].nodes[n]['text'] for n in c['G_orig'].nodes()}
        dot_orig = generate_graphviz_incremental(c['G_orig'], c['level_orig'], node_text_map_orig, None, set(c['G_orig'].nodes()))
        dot_pipe = generate_graphviz_incremental(c['G_pipe'], c['level_pipe'], c['node_text_map_pipe'], c['dest_changed_map_pipe'], visible_pipe_nodes, highlight_node, True)

        out_l, out_r = widgets.Output(), widgets.Output()
        out_l.add_class('responsive-anim-graph')
        out_r.add_class('responsive-anim-graph')

        with out_l: display(dot_orig)
        with out_r: display(dot_pipe)

        # ALTERAÇÃO: Estilo unificado para aumentar o título e fixar margens limpas
        title_style = 'color: #ffffff; text-align: center; font-weight: bold; font-family: sans-serif; font-size: 18px; margin: 0 0 15px 0; padding: 0;'

        # ALTERAÇÃO: Adicionado align_items='flex-start' no HBox para parear o topo das colunas perfeitamente
        split_box = widgets.HBox([
            widgets.VBox([widgets.HTML(f'<h3 style="{title_style}">1. Grafo Original</h3>'), out_l], layout=widgets.Layout(width='50%')),
            widgets.VBox([widgets.HTML(f'<h3 style="{title_style}">2. Gabarito</h3>'), out_r], layout=widgets.Layout(width='50%'))
        ], layout=widgets.Layout(width='100%', align_items='flex-start'))
        display(split_box)

def on_graph_prev_clicked(b):
    global current_graph_idx
    if current_graph_idx > 0: current_graph_idx -= 1; render_current_graph_step()

def on_graph_next_clicked(b):
    global current_graph_idx
    if update_pipeline_cache() and current_graph_idx < len(pipeline_cache['nodes_order']) - 1:
        current_graph_idx += 1; render_current_graph_step()

def on_graph_final_clicked(b):
    global current_graph_idx
    if update_pipeline_cache(): current_graph_idx = len(pipeline_cache['nodes_order']) - 1; render_current_graph_step()

btn_graph_prev.on_click(on_graph_prev_clicked)
btn_graph_next.on_click(on_graph_next_clicked)
btn_graph_final.on_click(on_graph_final_clicked)

# =========================================================================
# 🗂️ MONTAGEM E INTEGRACÃO DA INTERFACE DE ABAS
# =========================================================================
row1_canvas = widgets.HBox([txt_node_instruction, slider_node_level, btn_add_canvas_node], layout=widgets.Layout(gap='15px'))
row2_canvas = widgets.HBox([txt_graph_code, btn_load_graph_code], layout=widgets.Layout(gap='15px'))
row3_canvas = widgets.HBox([drop_remove_node, btn_remove_canvas_node, btn_clear_canvas, btn_validate_canvas_graph], layout=widgets.Layout(gap='15px'))
panel_canvas_layout = widgets.VBox([
    widgets.VBox([row1_canvas, row2_canvas, row3_canvas], layout=widgets.Layout(padding='15px', background_color='#fafbfc', border='1px solid #cbd5e1', border_radius='12px', margin='10px 0')),
    canvas_output_plot
])

row1_gabarito = widgets.HBox([btn_graph_prev, btn_graph_next, btn_graph_final, lbl_graph_step], layout=widgets.Layout(gap='10px', margin='10px 0'))
panel_gabarito_layout = widgets.VBox([row1_gabarito, pipe_compare_output])

tab_interface = widgets.Tab()
tab_interface.children = [panel_canvas_layout, panel_gabarito_layout]
tab_interface.set_title(0, '✏️ Canvas Interativo')
tab_interface.set_title(1, '📖 Gabarito')

def on_tab_changed(change):
    if change['new'] == 1: render_current_graph_step()
    elif change['new'] == 0: render_canvas_split_view()

tab_interface.observe(on_tab_changed, names='selected_index')
dashboard_central = widgets.VBox([css_unificado, tab_interface])
display(dashboard_central)

update_remove_dropdown(); render_canvas_split_view()

### Gera código com soft. pipeline e original e compara a saída de ambos

In [ ]:
#@title Usuário Propõe Código (Preâmbulo, Loop e Epílogo) e tem gabarito { display-mode: "form" }
# =========================================================================
# ⚙️ PATCH DE CORREÇÃO: Inicialização Real DO Banco Flutuante (F)
# =========================================================================
def patched_run_full_program(self, instructions, R_init=None, F_init=None, mem=None, max_steps=500_000):
    R = dict(R_init) if R_init is not None else self.config.fresh_registers()
    F = dict(F_init) if F_init is not None else {k: v for k, v in self.config.fresh_registers().items() if k.startswith('f')}
    mem = mem if mem is not None else self.config.fresh_memory()
    labels = self.build_label_map(instructions)
    pc, steps = 0, 0
    label_visits = defaultdict(int)
    n = len(instructions)

    while 0 <= pc < n and steps < max_steps:
        instr = instructions[pc]
        steps += 1
        if instr.get('type') == 'label':
            pc += 1
            continue
        if instr.get('label'):
            label_visits[instr['label']] += 1

        cat = instr['category']
        if cat == 'branch':
            s1, s2, target = instr['operands']
            a, b, op = self._get_reg(s1, R, F), self._get_reg(s2, R, F), instr['op']
            taken = {'!=': a != b, '==': a == b, '<': a < b,
                     '>': a > b, '<=': a <= b, '>=': a >= b}[op]
            if taken and target in label_visits and label_visits[target] >= self.config.forced_loops:
                taken = False
            pc = labels[target] if taken else pc + 1
        elif cat == 'jump':
            (target,) = instr['operands']
            if target in label_visits and label_visits[target] >= self.config.forced_loops:
                pc = pc + 1
            else:
                pc = labels[target]
        else:
            self.execute_straightline(instr, R, F, mem)
            pc += 1
    return R, F, mem, label_visits

def patched_simulate_pipeline(self, stages_user, strides, total_iterations):
    ws = self.config.word_size
    mem_pipe = self.config.fresh_memory()
    F_pipe = {k: v for k, v in self.config.reg_init.items() if k.startswith('f')}
    max_lvl = max(stages_user.keys()) if stages_user else 0
    total_cycles = total_iterations + max_lvl

    for cycle in range(total_cycles):
        F_snap = dict(F_pipe)
        mem_snap = list(mem_pipe)
        pending_F, pending_mem = {}, {}

        for lvl in sorted(stages_user.keys(), reverse=True):
            it = cycle - lvl
            if not (0 <= it < total_iterations):
                continue
            R_it = {reg: self.config.reg_init.get(reg, 0) + it * step for reg, step in strides.items()}
            for instr in stages_user[lvl]:
                cat, ops, op = instr['category'], instr['operands'], instr['op']
                if cat == 'load':
                    dest, mem_op = ops
                    offset, base = MEM_RE.match(mem_op).groups()
                    addr = R_it.get(base, 0) + int(offset)
                    idx = addr // ws
                    if idx >= len(mem_pipe):
                        _ensure_mem_size(mem_pipe, idx)
                    val = mem_snap[idx] if idx < len(mem_snap) else float(idx)
                    pending_F[dest] = val
                elif cat == 'store':
                    src, mem_op = ops
                    offset, base = MEM_RE.match(mem_op).groups()
                    addr = R_it.get(base, 0) + int(offset)
                    idx = addr // ws
                    if idx >= len(mem_pipe):
                        _ensure_mem_size(mem_pipe, idx)
                    pending_mem[idx] = F_snap.get(src, pending_F.get(src, 0.0))
                elif cat == 'mov':
                    dest, src = ops
                    pending_F[dest] = F_snap.get(src, pending_F.get(src, 0.0))
                elif cat == 'r_type':
                    dest, s1, s2 = ops
                    a = F_snap.get(s1, pending_F.get(s1, 0.0))
                    b = F_snap.get(s2, pending_F.get(s2, 0.0))
                    pending_F[dest] = a * b if op == '*' else (a + b if op == '+' else a - b)
                elif cat == 'i_type':
                    dest, src, imm = ops
                    a = F_snap.get(src, pending_F.get(src, 0.0)) if src.startswith('f') else R_it.get(src, 0)
                    pending_F[dest] = a + int(imm) if op == '+' else a - int(imm)

        F_pipe.update(pending_F)
        for idx, val in pending_mem.items():
            mem_pipe[idx] = val
    return mem_pipe

ISASimulator.run_full_program = patched_run_full_program
PipelineValidator.simulate_pipeline = patched_simulate_pipeline


# =========================================================================
# ✏️ ABA 1 — WIDGETS DO SANDBOX (usuário propõe preâmbulo/loop/epílogo)
# =========================================================================
challenge_title = widgets.HTML("""
    <div style="background: linear-gradient(135deg, #2e1065 0%, #3b0764 100%); padding: 18px 24px; border-radius: 12px 14px 0 0; color: white; font-family: sans-serif;">
        <h3 style="margin: 0; font-size: 22px; font-weight: 600;"> Valide seu Próprio Pipeline</h3>
        <p style="margin: 4px 0 0 0; font-size: 13px; opacity: 0.85;">Proponha sua escala de código.</p>
    </div>
""")

user_preamble_input = widgets.Textarea(
    value='# Digite seu preambulo aqui\n',
    placeholder='Instruções do Preâmbulo...',
    description='Preâmbulo:',
    layout=widgets.Layout(width='98%', height='100px', margin='0 0 10px 0')
)

user_kernel_input = widgets.Textarea(
    value='# Digite seu miolo do loop aqui\n',
    placeholder='Instruções do Loop...',
    description='Loop:',
    layout=widgets.Layout(width='98%', height='100px', margin='0 0 10px 0')
)

user_epilogue_input = widgets.Textarea(
    value='# Digite seu epilogo aqui\n',
    placeholder='Instruções do Epílogo...',
    description='Epílogo:',
    layout=widgets.Layout(width='98%', height='100px', margin='0 0 15px 0')
)

btn_validate_challenge = widgets.Button(
    description='Validar Equivalência Semântica', icon='check-double',
    button_style='primary', layout=widgets.Layout(width='320px', height='42px')
)

challenge_output = widgets.Output(layout=widgets.Layout(margin='16px 0 0 0'))

def on_validate_challenge_clicked(b):
    with challenge_output:
        clear_output()

        if 'program_instructions' not in globals() or not program_instructions:
            print("⚠️ Código base não encontrado. Certifique-se de carregar as instruções na Célula 2.")
            return

        validator_oficial = PipelineValidator(program_instructions, machine_config)
        try:
            mem_gabarito, _, total_iterations = validator_oficial.compute_reference()
        except Exception as e:
            print(f"❌ Erro ao rodar o programa de referência: {e}")
            return

        # 🧠 --- SISTEMA CENTRAL DE INTEGRAÇÃO E AUTO-CÁLCULO DE OFFSETS ---
        G_orig = build_dependency_graph(program_instructions)
        no_dep = [n for n in G_orig.nodes() if G_orig.degree(n) == 0]
        G_orig.remove_nodes_from(no_dep)
        level_orig = compute_layered_positions(G_orig)
        _, level_pipe, node_text_map_pipe, _ = build_pipelined_with_moves(G_orig, level_orig)

        def normalize_mem_instruction(text):
            text = text.split('#')[0].strip().lower()
            text_normalized = re.sub(r'(-?\d+)\((\w+)\)', r'(\2)', text)
            text_normalized = re.sub(r'\s*,\s*', ',', text_normalized)
            text_normalized = re.sub(r'\s+', ' ', text_normalized)
            return text_normalized

        # Mapeia a assinatura estrutural de cada instrução gerada para o seu nível real
        signature_to_level = {}
        for node, lvl in level_pipe.items():
            sig = normalize_mem_instruction(node_text_map_pipe[node])
            signature_to_level[sig] = lvl

        strides = ISASimulator.detect_induction_strides(program_instructions)

        def auto_adjust_line_offsets(line):
            line_clean = line.split('#')[0].strip()
            if not line_clean:
                return line

            sig = normalize_mem_instruction(line_clean)
            if sig in signature_to_level:
                lvl = signature_to_level[sig]
                mm = re.search(r'(-?\d+)\((\w+)\)', line, flags=re.IGNORECASE)
                if mm:
                    offset, base = mm.groups()
                    stride = strides.get(base.lower(), 0)
                    adjusted_offset = int(offset) - lvl * stride
                    return re.sub(r'-?\d+\(' + re.escape(base) + r'\)', f"{adjusted_offset}({base})", line, flags=re.IGNORECASE)
            return line
        # -----------------------------------------------------------------

        def clean_and_patch_block(text):
            lines_output = []
            for line in text.split('\n'):
                patched_line = auto_adjust_line_offsets(line)
                clean_content = patched_line.split('#')[0].strip()
                if clean_content:
                    lines_output.append(clean_content)
            return lines_output

        pre_lines = clean_and_patch_block(user_preamble_input.value)
        ker_lines = clean_and_patch_block(user_kernel_input.value)
        epi_lines = clean_and_patch_block(user_epilogue_input.value)

        has_loop_control = any(':' in line or any(branch_op in line for branch_op in ['bne', 'beq', 'bge', 'ble', 'bgt', 'blt', 'j']) for line in ker_lines)

        if has_loop_control:
            user_program_text = "\n".join(pre_lines + ker_lines + epi_lines)
        else:
            user_program_text = "\n".join(pre_lines + (ker_lines * total_iterations) + epi_lines)

        try:
            user_parsed_instructions = parse_program(user_program_text)
            simulador_sandbox = ISASimulator(machine_config)

            R_init_sandbox = dict(machine_config.reg_init)
            F_init_sandbox = {k: v for k, v in machine_config.reg_init.items() if k.startswith('f')}

            _, _, mem_usuario, _ = simulador_sandbox.run_full_program(
                user_parsed_instructions,
                R_init=R_init_sandbox,
                F_init=F_init_sandbox,
            )
        except Exception as e:
            print(f"❌ Erro de parsing ou execução no seu código proposto: {e}")
            return

        th_style = "padding: 10px; background-color: #1e293b; color: #f1f5f9; font-size: 13px;"
        td_style = "padding: 8px 10px; border-bottom: 1px solid #1e293b; font-family: monospace; font-size: 13px;"

        table_rows = ""
        mismatches_count = 0
        n_show = min(30, machine_config.mem_size)

        for i in range(n_show):
            v_init = float(i)
            v_gab = mem_gabarito[i]
            v_usr = mem_usuario[i]

            is_modified = (v_gab != v_init)
            row_bg = "background-color: #020617;" if is_modified else ""

            if v_gab == v_usr:
                status = "✔ Correto"
                status_color = "#4ade80"
            else:
                status = "❌ Erro"
                status_color = "#f87171"
                mismatches_count += 1

            table_rows += f"""
            <tr style="{row_bg}">
                <td style="{td_style} color: #94a3b8; font-weight: bold;">Índice [{i}]</td>
                <td style="{td_style} color: #64748b;">{v_init:.1f}</td>
                <td style="{td_style} color: #38bdf8;">{v_gab:.1f}</td>
                <td style="{td_style} color: #e2e8f0;">{v_usr:.1f}</td>
                <td style="{td_style} color: {status_color}; font-weight: bold;">{status}</td>
            </tr>
            """

        if mismatches_count == 0:
            veredicto_html = f"""
            <div style="margin-top: 16px; padding: 16px; background-color: #064e3b; border-left: 4px solid #10b981; color: #34d399; border-radius: 6px; font-family: sans-serif; font-size: 14px;">
                🎉 <b>Veredicto: PIPELINE APROVADO!</b> Seu arranjo de instruções respeitou todas as dependências temporais. O resultado final gerado na simulação é perfeitamente equivalente ao sequencial para {total_iterations} iterações.
            </div>
            """
        else:
            veredicto_html = f"""
            <div style="margin-top: 16px; padding: 16px; background-color: #4c0519; border-left: 4px solid #f43f5e; color: #f43f5e; border-radius: 6px; font-family: sans-serif; font-size: 14px;">
                ❌ <b>Veredicto: FALHA DE EQUIVALÊNCIA!</b> O simulador detectou {mismatches_count} divergências de dados na memória. Revise a ordem dos seus estágios ou a lógica de encaminhamento dos MOVs.
            </div>
            """

        html_view = widgets.HTML(f"""
            <h4 style="color: #f1f5f9; font-family: sans-serif; font-weight: 600; font-size: 18px; margin-bottom: 12px;">📊 Verificação de Impacto no Vetor de Dados (Configuração Ativa)</h4>
            <div style="background-color: #0f172a; padding: 12px; border-radius: 8px; max-height: 400px; overflow-y: auto; width: 100%;">
                <table style="width: 100%; border-collapse: collapse; text-align: left; font-family: sans-serif;">
                    <thead>
                        <tr>
                            <th style="{th_style}">Vetor</th>
                            <th style="{th_style}">Estado Inicial</th>
                            <th style="{th_style}">Gabarito Sequencial</th>
                            <th style="{th_style}">Seu Pipeline</th>
                            <th style="{th_style}">Resultado</th>
                        </tr>
                    </thead>
                    <tbody>{table_rows}</tbody>
                </table>
            </div>
            {veredicto_html}
        """, layout=widgets.Layout(width='100%'))

        display(html_view)

btn_validate_challenge.on_click(on_validate_challenge_clicked)

user_preamble_input.add_class('widget-textarea')
user_kernel_input.add_class('widget-textarea')
user_epilogue_input.add_class('widget-textarea')

form_box = widgets.VBox([
    challenge_title,
    widgets.VBox([
        user_preamble_input,
        user_kernel_input,
        user_epilogue_input,
        widgets.HBox([btn_validate_challenge], layout=widgets.Layout(align_items='center'))
    ], layout=widgets.Layout(padding='20px', border='1px solid #dcdde1', border_radius='0 0 12px 14px', background_color='#fafbfc'))
])


# =========================================================================
# 📖 ABA 2 — WIDGETS DO GABARITO (código gerado automaticamente)
# =========================================================================
asm_compare_button = widgets.Button(
    description='Gerar Código e Simular', icon='play',
    button_style='success', layout=widgets.Layout(width='380px', height='40px')
)
asm_compare_output = widgets.Output(layout=widgets.Layout(margin='14px 0 0 0'))

def extract_dest_reg_local(text):
    text = text.split('#')[0].strip().lower()
    parts = text.split(None, 1)
    if len(parts) < 2: return None
    opcode, rest = parts
    if opcode in ['sd', 'sw', 'sb', 'sh', 'bne', 'beq', 'bgt', 'blt']:
        return None
    subparts = rest.split(',')
    if subparts:
        possible_reg = subparts[0].strip()
        if re.match(r'^f\d+$', possible_reg):
            return possible_reg
    return None

def build_pipelined_with_moves(G_orig, level_orig):
    all_regs = []
    for node in G_orig.nodes():
        all_regs.extend(re.findall(r'[fF]\d+', G_orig.nodes[node]['text']))
    reg_indices = [int(r[1:]) for r in all_regs if r[0].lower() == 'f']
    max_reg_idx = max(reg_indices) if reg_indices else 0
    global_next_reg = max_reg_idx + 1

    G_pipe = nx.DiGraph()
    level_pipe, node_text_map, dest_changed_map = {}, {}, {}
    levels_to_nodes = defaultdict(list)
    for node, lvl in level_orig.items():
        levels_to_nodes[lvl].append(node)

    reg_name_at_level = defaultdict(dict)
    mov_nodes, move_node_counter, seen_destinations = {}, 1000, set()

    prod_consumers = defaultdict(list)
    for u, v, data in G_orig.edges(data=True):
        for r in data['regs']:
            prod_consumers[(u, r.lower())].append(v)

    for u in G_orig.nodes():
        orig_text = G_orig.nodes[u]['text']
        d = extract_dest_reg_local(orig_text)
        if d: reg_name_at_level[(u, d.lower())][level_orig[u]] = d

    prefer_upper = any(r.isupper() for r in all_regs)
    f_prefix = 'F' if prefer_upper else 'f'
    mov_opcode = 'MOV' if prefer_upper else 'mov'

    for lvl in sorted(levels_to_nodes.keys()):
        for (p, r), consumers in list(prod_consumers.items()):
            L_p = level_orig[p]
            L_end = max(level_orig[c] for c in consumers)
            if L_p < lvl <= L_end:
                if lvl < L_end:
                    mov_id = move_node_counter
                    move_node_counter += 1
                    r_prev = reg_name_at_level[(p, r.lower())][lvl - 1]
                    r_new = f"{f_prefix}{global_next_reg}"
                    global_next_reg += 1

                    node_text_map[mov_id] = f"{mov_opcode} {r_new}, {r_prev}"
                    level_pipe[mov_id] = lvl
                    dest_changed_map[mov_id] = True
                    G_pipe.add_node(mov_id)
                    mov_nodes[(p, r.lower(), lvl)] = mov_id
                    reg_name_at_level[(p, r.lower())][lvl] = r_new
                    parent = p if lvl - 1 == L_p else mov_nodes[(p, r.lower(), lvl - 1)]
                    G_pipe.add_edge(parent, mov_id, regs={r_prev})
                else:
                    reg_name_at_level[(p, r.lower())][lvl] = reg_name_at_level[(p, r.lower())][lvl - 1]

        for u in levels_to_nodes[lvl]:
            orig_text = G_orig.nodes[u]['text']
            src_mappings = {}
            for p_node in G_orig.predecessors(u):
                for r in G_orig[p_node][u]['regs']:
                    src_mappings[r.lower()] = reg_name_at_level[(p_node, r.lower())][lvl - 1]

            orig_dest = extract_dest_reg_local(orig_text)
            new_dest = orig_dest
            dest_changed = False
            if orig_dest:
                orig_dest_lower = orig_dest.lower()
                if orig_dest_lower in seen_destinations:
                    new_dest = f"{f_prefix}{global_next_reg}"
                    global_next_reg += 1
                    dest_changed = True
                seen_destinations.add(orig_dest_lower)

            parts = orig_text.split(None, 1)
            if len(parts) == 2:
                opcode, rest = parts
                subparts = rest.split(',')
                is_store = opcode.lower() in ['sd', 'sw', 'sb', 'sh']
                new_subparts = []
                for idx, sub in enumerate(subparts):
                    sub_clean = sub.strip()
                    if idx == 0 and not is_store and orig_dest is not None:
                        new_subparts.append(new_dest)
                    else:
                        updated_sub = sub_clean
                        for orig_src, mapped_name in src_mappings.items():
                            updated_sub = re.sub(r'\b' + re.escape(orig_src) + r'\b', mapped_name, updated_sub, flags=re.IGNORECASE)
                        new_subparts.append(updated_sub)

                if opcode.lower() == 'mv':
                    opcode = mov_opcode

                updated_text = f"{opcode} {', '.join(new_subparts)}"
            else:
                updated_text = orig_text

            node_text_map[u] = updated_text
            level_pipe[u] = lvl
            dest_changed_map[u] = dest_changed
            G_pipe.add_node(u)

            if orig_dest: reg_name_at_level[(u, orig_dest.lower())][lvl] = new_dest
            for p_node in G_orig.predecessors(u):
                for r in G_orig[p_node][u]['regs']:
                    parent_in_pipe = p_node if lvl - 1 == level_orig[p_node] else mov_nodes[(p_node, r.lower(), lvl - 1)]
                    G_pipe.add_edge(parent_in_pipe, u, regs={reg_name_at_level[(p_node, r.lower())][lvl - 1]})

    return G_pipe, level_pipe, node_text_map, dest_changed_map

# 🧠 RENDERIZA O TEXTO MANTENDO SEMPRE OS OFFSETS ORIGINAIS LIMPOS (0(r1), 4(r1), etc.)
def generate_asm_text_blocks(level_pipe, node_text_map_pipe, program_instructions):
    if not level_pipe:
        vazio = "    # (nenhuma instrução escalonada)"
        return vazio, vazio, vazio

    max_level = max(level_pipe.values())

    stages = defaultdict(list)
    for node, lvl in level_pipe.items():
        # Repassa o texto puro gerado direto, mantendo o offset original intacto para o usuário
        adjusted_text = node_text_map_pipe[node]
        stages[lvl].append(adjusted_text)

    def block_for_levels(levels_subset):
        lines = []
        for lvl in sorted(levels_subset, reverse=True):
            lines.extend(stages.get(lvl, []))
        return lines

    ctrl_instructions = [
        inst['raw'] for inst in program_instructions
        if inst.get('category') in ('branch', 'jump', 'i_type')
    ]
    label_name = next((inst['label'] for inst in program_instructions if inst.get('label')), None) or 'LOOP'

    preamble_lines = []
    for k in range(max_level):
        preamble_lines.append(f"    # -- ciclo de enchimento {k + 1}/{max_level} --")
        preamble_lines.extend(f"    {line}" for line in block_for_levels(range(0, k + 1)))
    preamble_text = "\n".join(preamble_lines) if preamble_lines else "    # (pipeline raso — sem preâmbulo necessário)"

    kernel_lines = [f"{label_name}:"]
    kernel_lines.extend(f"    {line}" for line in block_for_levels(stages.keys()))
    kernel_lines.extend(f"    {ctrl}" for ctrl in ctrl_instructions)
    kernel_text = "\n".join(kernel_lines)

    epilogue_lines = []
    for k in range(1, max_level + 1):
        epilogue_lines.append(f"    # -- ciclo de esvaziamento {k}/{max_level} --")
        epilogue_lines.extend(f"    {line}" for line in block_for_levels(range(k, max_level + 1)))
    epilogue_text = "\n".join(epilogue_lines) if epilogue_lines else "    # (pipeline raso — sem epílogo necessário)"

    return preamble_text, kernel_text, epilogue_text

def on_asm_compare_clicked(b):
    with asm_compare_output:
        clear_output()

        if 'program_instructions' not in globals() or not program_instructions:
            print("⚠️ Nenhuma instrução encontrada no programa. Adicione instruções na primeira célula.")
            return

        G_orig = build_dependency_graph(program_instructions)
        no_dep = [n for n in G_orig.nodes() if G_orig.degree(n) == 0]
        G_orig.remove_nodes_from(no_dep)

        if G_orig.number_of_nodes() == 0:
            print("ℹ️ Nenhuma dependência de registradores para processar.")
            return

        level_orig = compute_layered_positions(G_orig)
        G_pipe, level_pipe, node_text_map_pipe, dest_changed_map_pipe = build_pipelined_with_moves(G_orig, level_orig)

        pre_text, kernel_text, epi_text = generate_asm_text_blocks(level_pipe, node_text_map_pipe, program_instructions)

        orig_insts_list = [G_orig.nodes[n]['text'] for n in sorted(G_orig.nodes())]
        orig_text = "LOOP_ORIGINAL:\n" + "\n".join(f"    {inst}" for inst in orig_insts_list)

        generated_pipeline_nodes = [{'text': node_text_map_pipe[n], 'level': level_pipe[n]} for n in G_pipe.nodes()]

        validator = PipelineValidator(program_instructions, machine_config)
        result = validator.validate(generated_pipeline_nodes)

        if result.get("error"):
            print(f"❌ {result['error']}")
            return

        code_style = (
            "background-color: #0f172a; color: #38bdf8; padding: 16px; "
            "border-radius: 10px; font-family: 'Consolas', monospace; "
            "font-size: 14px; line-height: 1.6; overflow-x: auto; white-space: pre;"
        )

        html_original = widgets.HTML(f"""
            <h4 style="color: #f1f5f9; font-family: sans-serif; margin-bottom: 8px; font-weight: 600; font-size: 20px;">🔄 Código Original (Sequencial)</h4>
            <div style="{code_style} color: #e2e8f0;">{orig_text}</div>
        """, layout=widgets.Layout(width='35%'))

        html_pipelined = widgets.HTML(f"""
            <h4 style="color: #f1f5f9; font-family: sans-serif; margin-bottom: 8px; font-weight: 600; font-size: 20px;">🚀 Código Otimizado Gerado com Offsets Reais</h4>
            <div style="{code_style} margin-bottom: 12px; border-left: 4px solid #f59e0b; color: #fbbf24;">{pre_text}</div>
            <div style="{code_style} margin-bottom: 12px; border-left: 4px solid #10b981; color: #34d399;">{kernel_text}</div>
            <div style="{code_style} border-left: 4px solid #ef4444; color: #f87171;">{epi_text}</div>
        """, layout=widgets.Layout(width='62%'))

        layout_codigo = widgets.HBox([html_original, html_pipelined], layout=widgets.Layout(gap='20px', width='100%'))

        th_style = "padding: 10px; background-color: #1e293b; color: #f1f5f9; position: sticky; top: 0; font-size: 13px;"
        td_style = "padding: 8px 10px; border-bottom: 1px solid #1e293b; font-family: monospace; font-size: 13px;"

        table_rows = ""
        mem_gabarito = result["mem_gabarito"]
        mem_pipe = result["mem_pipe"]
        n_show = min(30, machine_config.mem_size)

        for i in range(n_show):
            v_init = float(i)
            v_mem = mem_gabarito[i]
            v_l = mem_pipe[i]

            is_modified = (v_mem != v_init)
            row_bg = "background-color: #111827;" if is_modified else ""
            idx_style = "color: #38bdf8; font-weight: bold;" if is_modified else "color: #94a3b8;"
            status = "✔ Perfeito" if v_mem == v_l else "❌ Mismatch"
            status_color = "#4ade80" if v_mem == v_l else "#f87171"

            table_rows += f"""
            <tr style="{row_bg} border-bottom: 1px solid #1e293b;">
                <td style="{td_style} {idx_style}">Índice [{i}]</td>
                <td style="{td_style} color: #64748b;">{v_init}</td>
                <td style="{td_style} color: #38bdf8;">{v_mem:.1f}</td>
                <td style="{td_style} color: #34d399;">{v_l:.1f}</td>
                <td style="{td_style} color: {status_color}; font-weight: bold;">{status}</td>
            </tr>
            """

        status_veredicto = "✔ <b>Sucesso na Homologação:</b> O estado final bateu 100% idêntico!" if result["success"] else "❌ <b>Falha na Homologação:</b> Ocorreu divergência matemática."
        bg_veredicto = "#14532d" if result["success"] else "#4c0519"
        color_veredicto = "#4ade80" if result["success"] else "#f43f5e"

        html_tables = widgets.HTML(f"""
            <hr style="border: 0; border-top: 1px solid #334155; margin: 24px 0;">
            <h4 style="color: #f1f5f9; font-family: sans-serif; font-weight: 600; font-size: 20px; margin-bottom: 4px;">📊 Comparação Semântica Unificada (Célula 3)</h4>
            <div style="background-color: #0f172a; padding: 12px; border-radius: 8px; max-height: 450px; overflow-y: auto; width: 100%;">
                <table style="width: 100%; border-collapse: collapse; text-align: left; font-family: sans-serif;">
                    <thead>
                        <tr>
                            <th style="{th_style}">Posição do Vetor</th>
                            <th style="{th_style}">Valor Inicial</th>
                            <th style="{th_style}">Loop Original (Gabarito)</th>
                            <th style="{th_style}">Código Otimizado Automático</th>
                            <th style="{th_style}">Status de Validação</th>
                        </tr>
                    </thead>
                    <tbody>{table_rows}</tbody>
                </table>
            </div>
            <div style="margin-top: 16px; padding: 14px; background-color: {bg_veredicto}; color: {color_veredicto}; border-radius: 6px; font-family: sans-serif; font-size: 14px; font-weight: 500;">
                {status_veredicto}
            </div>
        """, layout=widgets.Layout(width='100%'))

        display(widgets.VBox([layout_codigo, html_tables]))

asm_compare_button.on_click(on_asm_compare_clicked)
asm_compare_output.add_class('output-box')


# =========================================================================
# 🗂️ MONTAGEM FINAL: AS 2 ABAS, UMA ÚNICA CÉLULA, UM ÚNICO display()
# =========================================================================
css_unificado_asm = widgets.HTML("""
    <style>
    .lm-TabBar-tab, .p-TabBar-tab {
        width: auto !important; min-width: 220px !important;
        padding: 0 20px !important; overflow: visible !important;
    }
    .lm-TabBar-tabLabel, .p-TabBar-tabLabel {
        color: #ffffff !important; font-weight: 600 !important;
        font-size: 13px !important; overflow: visible !important; text-overflow: clip !important;
    }
    .lm-TabBar-tab:not(.lm-mod-current), .p-TabBar-tab:not(.p-mod-current) {
        background-color: #334155 !important; opacity: 0.7;
    }
    .lm-TabBar-tab.lm-mod-current, .p-TabBar-tab.p-mod-current {
        background-color: #1e293b !important; border-bottom: 2px solid #38bdf8 !important; opacity: 1;
    }
    </style>
    <div style="background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%); padding: 18px 24px; border-radius: 12px 14px 0 0; color: white; font-family: sans-serif;">
        <h3 style="margin: 0; font-size: 22px; font-weight: 600;">🧵 Central de Código: Preâmbulo, Loop e Epílogo</h3>
        <p style="margin: 4px 0 0 0; font-size: 13px; opacity: 0.85;">Proponha seu próprio código escalonado no Sandbox ou veja o Gabarito gerado automaticamente a partir do grafo.</p>
    </div>
""")

panel_sandbox_layout = widgets.VBox([form_box, challenge_output])
panel_gabarito_asm_layout = widgets.VBox([
    widgets.HBox([asm_compare_button], layout=widgets.Layout(margin='10px 0')),
    asm_compare_output
])

tab_interface_asm = widgets.Tab()
tab_interface_asm.children = [panel_sandbox_layout, panel_gabarito_asm_layout]
tab_interface_asm.set_title(0, '✏️ Proponha seu Código')
tab_interface_asm.set_title(1, '📖 Gabarito')

def on_tab_changed_asm(change):
    if change['new'] == 1:
        on_asm_compare_clicked(None)

tab_interface_asm.observe(on_tab_changed_asm, names='selected_index')

dashboard_central_asm = widgets.VBox([css_unificado_asm, tab_interface_asm])
display(dashboard_central_asm)

### Ciclos com Escalonamento Dinâmico

In [ ]:
#@title Interface para visualizar os ciclos usando abordagem ingênua, com soft. pipeline gabarito e o soft. pipeline do usuário  { display-mode: "form" }

# =========================================================================
# ⚙️ GERENCIADOR DE HISTÓRICO DE PASSOS
# =========================================================================
current_sim = None
history_stack = []

drop_select_stream = widgets.Dropdown(
    options=[
        ("🏆 Com Soft. Pipeline (Seu Grafo)", 'canvas'),
        ("⚡ Com Soft. Pipeline (Gabarito)", 'gabarito'),
        ("🔄 Loop Original (Sequencial)", 'original')
    ],
    value='canvas',
    description='Código Alvo:',
    layout=widgets.Layout(width='320px')
)

btn_reset_sim = widgets.Button(description='Reiniciar', icon='sync', button_style='warning', layout=widgets.Layout(width='120px', height='40px'))
btn_prev_pc = widgets.Button(description='PC - 1', icon='arrow-left', button_style='danger', layout=widgets.Layout(width='110px', height='40px'))
btn_step_pc = widgets.Button(description='PC + 1', icon='arrow-right', button_style='info', layout=widgets.Layout(width='110px', height='40px'))
btn_final_pc = widgets.Button(description='Ir para o Final', icon='fast-forward', button_style='success', layout=widgets.Layout(width='150px', height='40px'))

dashboard_panel_output = widgets.Output(layout=widgets.Layout(margin='14px 0 0 0'))

def setup_chosen_simulation(*_):
    global current_sim, history_stack
    history_stack = []

    with dashboard_panel_output:
        clear_output()
        if 'program_instructions' not in globals() or not program_instructions:
            print("⚠️ Nenhuma instrução base encontrada. Carregue o conjunto de instruções requerido primeiro!")
            return

        selected = drop_select_stream.value
        if selected == 'original':
            stream = [inst['raw'] for inst in program_instructions]
        elif selected == 'gabarito':
            G_orig = build_dependency_graph(program_instructions)
            no_dep = [n for n in G_orig.nodes() if G_orig.degree(n) == 0]; G_orig.remove_nodes_from(no_dep)
            level_orig = compute_layered_positions(G_orig)

            _, level_pipe, node_text_map_pipe, _, _ = build_pipelined_with_moves_ordered(G_orig, level_orig)

            stages = defaultdict(list)
            for node, lvl in level_pipe.items(): stages[lvl].append(node_text_map_pipe[node])
            stream = []

            # CORREÇÃO 1: Adicionado reverse=True para espelhar o Slide 9 (sd -> mult -> add -> ld)
            for lvl in sorted(stages.keys(), reverse=True):
                for inst in stages[lvl]: stream.append(inst)

            for ctrl in [inst['raw'] for inst in program_instructions if inst['category'] in ('branch', 'jump', 'i_type')]: stream.append(ctrl)
        else:
            if not user_canvas_nodes:
                print("ℹ️ Seu Canvas está em branco. Adicione nós ou selecione 'Gabarito' / 'Original'.")
                return

            # CORREÇÃO 2: Adicionado reverse=True para que o Canvas do usuário também simule na ordem correta do pipeline
            canvas_nodes_sorted = sorted(user_canvas_nodes, key=lambda x: x['level'], reverse=True)
            stream = [item['text'] for item in canvas_nodes_sorted]
            for ctrl in [inst['raw'] for inst in program_instructions if inst['category'] in ('branch', 'jump', 'i_type')]: stream.append(ctrl)

        current_sim = TomasuloEngineInteractive(stream)
        btn_step_pc.disabled = False
        btn_final_pc.disabled = False
        render_dashboard_view()

def step_simulation_click(b):
    global current_sim, history_stack
    if current_sim is None or current_sim.is_done(): return

    history_stack.append(copy.deepcopy(current_sim))
    current_sim.step_cycle()
    current_sim.clk += 1
    render_dashboard_view()

def prev_simulation_click(b):
    global current_sim, history_stack
    if not history_stack: return

    current_sim = history_stack.pop()
    btn_step_pc.disabled = False
    btn_final_pc.disabled = False
    render_dashboard_view()

def jump_to_final_click(b):
    global current_sim, history_stack
    if current_sim is None or current_sim.is_done(): return

    with dashboard_panel_output:
        while not current_sim.is_done() and current_sim.clk < 1000:
            history_stack.append(copy.deepcopy(current_sim))
            current_sim.step_cycle()
            current_sim.clk += 1
        render_dashboard_view()

def render_dashboard_view():
    with dashboard_panel_output:
        clear_output(wait=True)
        if current_sim is None: return

        print(f"⏱️ CLOCK ATUAL: {current_sim.clk - 1}\n")

        table_list = []
        for instruction, stages in current_sim.table_data.items():
            display_name = instruction.split('   #')[0]
            row = [display_name] + [stages[h] for h in current_sim.x_headers]
            table_list.append(row)

        headers_display = ["Instruction"] + [h.capitalize() if "_" not in h else h[:h.find("_")].capitalize() + " (It2)" for h in current_sim.x_headers]
        print(tabulate(table_list, headers_display, tablefmt="fancy_grid"))

        if current_sim.is_done():
            print(f"\n🛑 SIMULAÇÃO CONCLUÍDA: O teto limite da 2ª iteração foi alcançado!")
            btn_step_pc.disabled = True
            btn_final_pc.disabled = True

btn_reset_sim.on_click(setup_chosen_simulation)
btn_prev_pc.on_click(prev_simulation_click)
btn_step_pc.on_click(step_simulation_click)
btn_final_pc.on_click(jump_to_final_click)
drop_select_stream.observe(setup_chosen_simulation, names='value')

controls_bar = widgets.HBox([drop_select_stream, btn_reset_sim, btn_prev_pc, btn_step_pc, btn_final_pc], layout=widgets.Layout(gap='10px', margin='10px 0'))
display(controls_bar, dashboard_panel_output)

setup_chosen_simulation()

Output(layout=Layout(margin='14px 0 0 0'))